In [47]:
# =============================================================================
# balanced_ppo.ipynb
# EGX30 Balanced PPO Portfolio Optimizer
#
# Architecture: Shared-weight MLP encoder + Self-attention + PPO (SB3)
#
# Pipeline:
#   PART 1 — Sanity checks & artifact loading
#   PART 2 — Architecture, Environment, Training, Evaluation
# =============================================================================

# %%

In [48]:
from pathlib import Path
import warnings
import json
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F_torch

from scipy import stats
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import (
    EvalCallback,
    StopTrainingOnNoModelImprovement,
    BaseCallback,
)
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# %%

In [49]:
ROOT = Path(".").resolve()

OHLCV_DIR          = ROOT / "data/OHLCV"
MACRO_SIGNALS_DIR  = ROOT / "data/macro_signals"
TENSORS_DIR        = ROOT / "data/tensors"
PREPROCESSING_DIR  = ROOT / "data/preprocessing"
MODEL_INPUTS_DIR   = ROOT / "data/model_inputs"
MODEL_VERSIONS_DIR = ROOT / "model_versions"

LGBM_PRED_PATH             = MODEL_INPUTS_DIR  / "lgbm_predictions.csv"
DAILY_CLASSIFICATIONS_PATH = MODEL_INPUTS_DIR  / "daily_risk_classifications.csv"
CONIA_DATA_PATH            = MACRO_SIGNALS_DIR / "conia_o_n_rate.xlsx"
INFLATION_DATA_PATH        = MACRO_SIGNALS_DIR / "inflation_rate.xlsx"
MARKET_DATA_PATH           = MACRO_SIGNALS_DIR / "market_data.csv"

for d in [TENSORS_DIR, PREPROCESSING_DIR, MACRO_SIGNALS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# %%

In [50]:
VERSION      = "v7_higher_cash_penalty"
RISK_PROFILE = "balanced"

VERSION_DIR = MODEL_VERSIONS_DIR / VERSION
VERSION_DIR.mkdir(parents=True, exist_ok=True)

# %%

In [51]:
try:
    from rrfr_fetcher import get_real_risk_free_rate
    _fetched_rrfr = get_real_risk_free_rate()
    ANNUAL_RISK_FREE_RATE = _fetched_rrfr if _fetched_rrfr is not None else 0.21
    _rrfr_source = "live" if _fetched_rrfr is not None else "fallback (fetch returned None)"
except Exception as _e:
    ANNUAL_RISK_FREE_RATE = 0.21
    _rrfr_source = f"fallback (import/fetch error: {_e})"

# Log-domain risk-free rates — consistent with log-return arithmetic throughout
ANN_LOG_RFR    = np.log(1 + ANNUAL_RISK_FREE_RATE)
print(f"Annual risk-free rate : {ANNUAL_RISK_FREE_RATE:.4%}  [{_rrfr_source}]")
print(f"Annual log RFR        : {ANN_LOG_RFR:.6f}")

# %%

2026-06-27 00:47:00,032 - INFO - CONIA: stale (cached 2026-06-25, need 2026-06-24)
2026-06-27 00:47:00,035 - INFO - ====== WebDriver manager ======


2026-06-27 00:47:01,331 - INFO - Get LATEST chromedriver version for google-chrome
2026-06-27 00:47:01,803 - INFO - Get LATEST chromedriver version for google-chrome
2026-06-27 00:47:02,215 - INFO - Get LATEST chromedriver version for google-chrome
2026-06-27 00:47:03,219 - INFO - WebDriver version 149.0.7827.155 selected
2026-06-27 00:47:03,226 - INFO - Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/149.0.7827.155/win32/chromedriver-win32.zip
2026-06-27 00:47:03,227 - INFO - About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/149.0.7827.155/win32/chromedriver-win32.zip
2026-06-27 00:47:03,714 - INFO - Driver downloading response is 200
2026-06-27 00:47:06,367 - INFO - Get LATEST chromedriver version for google-chrome
2026-06-27 00:47:07,399 - INFO - Driver has been saved in cache [C:\Users\mirae\.wdm\drivers\chromedriver\win64\149.0.7827.155]
2026-06-27 00:47:10,250 - INFO - Navigating to https://www.cbe.org.eg/en/

Annual risk-free rate : 8.2024%  [live]
Annual log RFR        : 0.078834


In [52]:
VOLUME_AVG_WINDOW  = 20
RETURN_CLAMP       = 0.25
CORR_WINDOW        = 20
MA_SHORT           = 20
MA_LONG            = 60
CONIA_DELTA_WINDOW = 60

FEATURE_NAMES = [
    "ret_1d",
    "ret_5d",
    "ret_20d",
    "lgbm_pred",
    "lgbm_pred_valid",
    "gk_vol_20d",
    "skew_20d",
    "kurt_20d",
    "rel_volume",
    "avg_pairwise_corr",
]

TRAIN_END = "2021-12-31"
VAL_END   = "2022-12-31"

# %%

In [53]:
ENCODER_INPUT_DIM  = len(FEATURE_NAMES) + 1   # 10 features + 1 current weight = 11
ENCODER_HIDDEN_DIM = 64
ENCODER_OUTPUT_DIM = 64
ATTENTION_HEADS    = 1
ATTENTION_DIM      = ENCODER_OUTPUT_DIM
POLICY_HIDDEN_DIMS = [256, 256]

USE_SOFT_MASK                  = False
SOFT_MASK_CONSERVATIVE_WEIGHT = 0.5
SOFT_MASK_AGGRESSIVE_WEIGHT   = 0.2

# Delta action parameters
# Action is interpreted as a position-relative adjustment rather than a target portfolio.
# delta_i = tanh(action_i) * max(current_weight_i, MIN_ENTRY_FLOOR)
# MIN_ENTRY_FLOOR gives stocks at zero weight a small seed so the policy can enter them.
# DELTA_SCALE controls max adjustment per step: 1.0 = can double or halve any position.
USE_DELTA_ACTION  = True
MIN_ENTRY_FLOOR   = 0.005   # 0.5% seed for stocks at zero — allows new position entry
DELTA_SCALE       = 1.0     # tanh output ∈ [-1, +1]; delta = tanh(a) * (w + floor)

# Minimum position size: positions below this threshold are zeroed and redistributed.
# Eliminates softmax dust and forces genuine allocation decisions.
MIN_POSITION_SIZE = 0.005   # 50bp — below this is economically meaningless

In [54]:
# =============================================================================
# CELL 7 — RL / training parameters + model assumptions
# Updated: rolling Sharpe window added to reward config (assumption [3] fix)
# =============================================================================
EPISODE_LENGTH        = 24
REBALANCE_DAYS        = 21
PERIODS_PER_YEAR      = 252 / 21
MAX_CASH_WEIGHT       = 0.30
MAX_STOCK_WEIGHT      = 0.20
MIN_STOCK_WEIGHT      = 0.0
BASE_TRANSACTION_COST = 0.0064
MAX_TRANSACTION_COST  = 0.015
TURNOVER_CAP          = 0.80
N_ENVS                = 4
BASE_TIMESTEPS        = 3_500_000
MAX_EVAL_EPISODES     = 50
N_BOOTSTRAP           = 2000
BOOTSTRAP_BLOCK       = 3
CONFIDENCE_LEVEL      = 0.95
HAC_LAGS              = 3

# Rolling Sharpe window for reward shaping (assumption [3] fix).
# Policy optimizes a local rolling Sharpe estimate rather than raw per-step
# return, narrowing the gap between reward signal and evaluation metric.
# Window = 6 rebalancing periods (~126 days). Requires at least 2 steps of
# history before the Sharpe component activates; pure return reward used
# until then. Set to 0 to disable (reverts to raw return reward).
SHARPE_REWARD_WINDOW = 6

try:
    from statsmodels.regression.linear_model import OLS
    from statsmodels.tools import add_constant
    _STATSMODELS_AVAILABLE = True
except ImportError:
    _STATSMODELS_AVAILABLE = False
    print("⚠ statsmodels not installed — HAC t-test will fall back to naive t-test.")
    print("  Install with: pip install statsmodels")

WALK_FORWARD_FOLDS = [
    {"name": "fold_1", "train_end": "2019-12-31", "val_end": "2020-12-31", "test_end": "2021-12-31"},
    {"name": "fold_2", "train_end": "2020-12-31", "val_end": "2021-12-31", "test_end": "2022-12-31"},
    {"name": "fold_3", "train_end": "2021-12-31", "val_end": "2022-12-31", "test_end": "2023-12-31"},
    {"name": "fold_4", "train_end": "2022-12-31", "val_end": "2023-12-31", "test_end": "2025-12-31"},
]

PPO_PARAMS = {
    "learning_rate": 3e-4,
    "n_steps":       2048,
    "batch_size":    64,
    "n_epochs":      10,
    "gamma":         0.995,
    "gae_lambda":    0.95,
    "clip_range":    0.2,
    "ent_coef":      0.03,
    "vf_coef":       0.5,
    "max_grad_norm": 0.5,
    "verbose":       0,
}

REWARD_CONFIGS = {
    "return_weight":       1.0,
    "sharpe_weight":       0.3,   # rolling Sharpe bonus — assumption [3] fix
    "turnover_penalty":    0.01,
    "downside_penalty":    0.4,
    "drawdown_penalty":    1.0,
    "hhi_penalty":         0.5,
    "cash_penalty":        4.5,
}

PALETTE = {
    "primary":   "#2c7bb6",
    "secondary": "#d7191c",
    "tertiary":  "#fdae61",
    "train":     "#2c7bb6",
    "val":       "#fdae61",
    "test":      "#2ca25f",
    "alert":     "#d7191c",
}

# =============================================================================
# Model assumptions — fixes documented
# =============================================================================
#
# [1] 21-day window compounding approximation — NOT FIXABLE without redesign
#     portfolio_return = dot(weights_t, sum(daily_log_returns, t→t+21))
#     Summing daily log returns is mathematically correct for a fixed-weight
#     portfolio (log-additivity holds). The residual mismatch is that weights
#     drift intra-window as prices move; fixing this would require daily
#     rebalancing simulation (daily step environment). Acknowledged as a
#     model limitation; not correctable in the current 21-day step design.
#
# [2] Cash timing option asymmetry — ADDRESSED
#     RL agent can hold up to MAX_CASH_WEIGHT (30%) cash; EQW is always
#     fully invested. Addressed by multi_episode_evaluate_no_cash() which
#     forces cash=0 post-hoc, and timing_option_sharpe in aggregate results
#     which decomposes total Sharpe margin into selection alpha + timing option.
#
# [3] Reward-evaluation objective gap — PARTIALLY FIXED
#     Previous reward: local penalized per-step log return.
#     Updated reward: weighted combination of per-step return and a rolling
#     Sharpe estimate over the last SHARPE_REWARD_WINDOW steps. The rolling
#     Sharpe term trains the policy to be aware of its own return volatility,
#     narrowing the gap with the Sharpe-based evaluation metric.
#     Residual gap: rolling Sharpe over 6 periods ≠ global CAGR/Sharpe over
#     the full episode. Full elimination would require episodic Sharpe reward
#     which is incompatible with per-step PPO training.
#
# [4] Projection cascade non-identifiability — PARTIALLY FIXED
#     Previous: 5 sequential nonlinear projections (delta → mask → cash cap
#     → position cap → min-position → turnover cap).
#     Updated: consolidated to 3 stages: (1) action → unconstrained proposed
#     weights, (2) single combined simplex projection enforcing all weight
#     constraints simultaneously, (3) turnover cap. Reduces the many-to-one
#     mapping and discontinuities introduced by sequential redistribution loops.
#     Residual: turnover cap still creates a discontinuous projection when
#     binding. Not eliminable without removing the cap entirely.
# =============================================================================

print(f"Configuration loaded.")
print(f"  VERSION          : {VERSION}")
print(f"  RISK_PROFILE     : {RISK_PROFILE}")
print(f"  USE_SOFT_MASK    : {USE_SOFT_MASK}")
print(f"  USE_DELTA_ACTION : {USE_DELTA_ACTION}")
print(f"  RFR (annual)     : {ANNUAL_RISK_FREE_RATE:.4%}")
print(f"  Encoder          : {ENCODER_INPUT_DIM}→{ENCODER_HIDDEN_DIM}→{ENCODER_OUTPUT_DIM}")
print(f"  Attention heads  : {ATTENTION_HEADS}")
print(f"  Policy head      : {POLICY_HIDDEN_DIMS}")
print(f"  ent_coef         : {PPO_PARAMS['ent_coef']}")
print(f"  sharpe_weight    : {REWARD_CONFIGS['sharpe_weight']}  (rolling window={SHARPE_REWARD_WINDOW})")
print(f"  HAC_LAGS         : {HAC_LAGS}")
print(f"\n  Assumptions [1] not fixable, [2] addressed, [3] partially fixed, [4] partially fixed.")

Configuration loaded.
  VERSION          : v7_higher_cash_penalty
  RISK_PROFILE     : balanced
  USE_SOFT_MASK    : False
  USE_DELTA_ACTION : True
  RFR (annual)     : 8.2024%
  Encoder          : 11→64→64
  Attention heads  : 1
  Policy head      : [256, 256]
  ent_coef         : 0.03
  sharpe_weight    : 0.3  (rolling window=6)
  HAC_LAGS         : 3

  Assumptions [1] not fixable, [2] addressed, [3] partially fixed, [4] partially fixed.


In [55]:
feat_check    = torch.load(TENSORS_DIR / "feature_tensor.pt",      weights_only=True)
ret_check     = torch.load(TENSORS_DIR / "returns_tensor.pt",      weights_only=True)
mask_check    = torch.load(TENSORS_DIR / "mask_tensor.pt",         weights_only=True)
profile_check = torch.load(TENSORS_DIR / "profile_mask_tensor.pt", weights_only=True)
signal_check  = torch.load(TENSORS_DIR / "market_signal.pt",       weights_only=True)
splits_check  = np.load(PREPROCESSING_DIR / "data_splits.npz")

T = feat_check.shape[0]

assert feat_check.ndim == 3
assert ret_check.shape   == feat_check.shape[:2]
assert mask_check.shape  == feat_check.shape[:2]
assert profile_check.shape == feat_check.shape[:2]
assert signal_check.shape  == (T, 3)
assert np.isfinite(feat_check.numpy()).all()
assert np.isfinite(ret_check.numpy()).all()
assert np.isfinite(mask_check.numpy()).all()
assert np.isfinite(profile_check.numpy()).all()
assert np.isfinite(signal_check.numpy()).all()

_tr = splits_check["train"].tolist()
_va = splits_check["val"].tolist()
_te = splits_check["test"].tolist()
assert _tr[1] == _va[0]
assert _va[1] == _te[0]
assert _te[1] == T
assert len(splits_check["dates"]) == T

print("All sanity checks passed.")
print(f"  feature_tensor  : {feat_check.shape}  dtype={feat_check.dtype}")
print(f"  returns_tensor  : {ret_check.shape}")
print(f"  mask_tensor     : {mask_check.shape}")
print(f"  profile_mask    : {profile_check.shape}  coverage={profile_check.float().mean()*100:.1f}%")
print(f"  market_signal   : {signal_check.shape}")
print(f"    egx30 range   : [{signal_check[:,0].min():.3f}, {signal_check[:,0].max():.3f}]")
print(f"    conia range   : [{signal_check[:,1].min():.3f}, {signal_check[:,1].max():.3f}]")
print(f"    infl  range   : [{signal_check[:,2].min():.3f}, {signal_check[:,2].max():.3f}]")
print(f"  data_splits.npz : {T} dates verified ✓")
print("\nPART 1 complete.")

# %%

All sanity checks passed.
  feature_tensor  : torch.Size([2647, 31, 10])  dtype=torch.float32
  returns_tensor  : torch.Size([2647, 31])
  mask_tensor     : torch.Size([2647, 31])
  profile_mask    : torch.Size([2647, 31])  coverage=31.1%
  market_signal   : torch.Size([2647, 3])
    egx30 range   : [-4.194, 3.423]
    conia range   : [-2.413, 6.619]
    infl  range   : [-3.551, 3.231]
  data_splits.npz : 2647 dates verified ✓

PART 1 complete.


In [56]:
feature_tensor      = torch.load(TENSORS_DIR / "feature_tensor.pt",      weights_only=False).numpy()
returns_tensor      = torch.load(TENSORS_DIR / "returns_tensor.pt",      weights_only=False).numpy()
mask_tensor         = torch.load(TENSORS_DIR / "mask_tensor.pt",         weights_only=False).numpy()
profile_mask_tensor = torch.load(TENSORS_DIR / "profile_mask_tensor.pt", weights_only=False).numpy()
market_signal       = torch.load(TENSORS_DIR / "market_signal.pt",       weights_only=False).numpy()

with open(PREPROCESSING_DIR / "market_signal_stats.json") as f:
    market_signal_stats = json.load(f)

_splits              = np.load(PREPROCESSING_DIR / "data_splits.npz")
date_index           = _splits["dates"].tolist()
_split_ranges        = {k: _splits[k].tolist() for k in ("train", "val", "test")}
global_split_indices = {k: list(range(*v)) for k, v in _split_ranges.items()}

tickers          = sorted([p.stem.upper() for p in OHLCV_DIR.glob("*.csv")])
universe_indices = list(range(len(tickers)))
REL_VOL_IDX      = FEATURE_NAMES.index("rel_volume")
T_TOTAL          = feature_tensor.shape[0]
date_to_idx      = {d: i for i, d in enumerate(date_index)}

assert market_signal.shape[1] == 3
assert not np.isnan(market_signal).any()
assert T_TOTAL == len(date_index)

print(f"Artifacts loaded.")
print(f"  feature_tensor  : {feature_tensor.shape}")
print(f"  market_signal   : {market_signal.shape}")
print(f"  date_index      : {len(date_index)} dates  ({date_index[0]} → {date_index[-1]})")
print(f"  splits          : train {_split_ranges['train']}  "
      f"val {_split_ranges['val']}  test {_split_ranges['test']}")
print(f"  tickers ({len(tickers)})")

# =============================================================================
# PART 2 — ARCHITECTURE, ENVIRONMENT, TRAINING, EVALUATION
# =============================================================================

# %%

Artifacts loaded.
  feature_tensor  : (2647, 31, 10)
  market_signal   : (2647, 3)
  date_index      : 2647 dates  (2015-07-06 → 2026-06-04)
  splits          : train [0, 1579]  val [1579, 1823]  test [1823, 2647]
  tickers (31)


In [57]:
N_STOCKS     = len(tickers)
OBS_DIM      = N_STOCKS * ENCODER_INPUT_DIM + N_STOCKS + 3
FEATURES_DIM = N_STOCKS * ENCODER_OUTPUT_DIM + 3

# Log-domain period risk-free rate — consistent with log return arithmetic
# Fix: use log RFR instead of arithmetic RFR to avoid mixing return conventions
PERIOD_LOG_RFR = ANN_LOG_RFR / PERIODS_PER_YEAR

print(f"OBS_DIM      : {OBS_DIM}  ({N_STOCKS}×{ENCODER_INPUT_DIM} + {N_STOCKS} mask + 3 macro)")
print(f"FEATURES_DIM : {FEATURES_DIM}  ({N_STOCKS}×{ENCODER_OUTPUT_DIM} + 3 macro)")
print(f"Period log RFR: {PERIOD_LOG_RFR:.6f}")

# %%

OBS_DIM      : 375  (31×11 + 31 mask + 3 macro)
FEATURES_DIM : 1987  (31×64 + 3 macro)
Period log RFR: 0.006569


In [58]:
class SharedStockEncoder(nn.Module):
    """
    Identical MLP applied to every stock's 10-dim input vector independently.
    Shared weights force the network to learn stock-agnostic feature rules
    rather than memorising ticker-specific patterns by input position.

    Input : [batch, N, 10]  (9 normalised features + 1 current portfolio weight)
    Output: [batch, N, 64]  (stock embeddings)
    """
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.Tanh(),
        )

    def forward(self, x):
        # x: [batch, N, D]
        batch, N, D = x.shape
        return self.net(x.reshape(batch * N, D)).reshape(batch, N, -1)


class StockAttention(nn.Module):
    """
    Single-head self-attention over stock embeddings with profile-mask gating.

    Hard mask (USE_SOFT_MASK=False):
        Ineligible stocks receive -inf before softmax → zero attention weight.
    Soft mask (USE_SOFT_MASK=True):
        log(mask_value) added as an additive bias; partially eligible stocks
        receive proportionally reduced but non-zero attention.

    Residual connection ensures each stock retains its own embedding signal.

    Input : embeddings   [batch, N, 64]
            profile_mask [batch, N]  ∈ {0,1} hard  or  {0, 0.2, 0.5, 1.0} soft
    Output: [batch, N, 64]
    """
    def __init__(self, embed_dim, n_heads=1):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.embed_dim = embed_dim
        self.n_heads   = n_heads
        self.head_dim  = embed_dim // n_heads
        self.scale     = self.head_dim ** -0.5
        self.q_proj    = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj    = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj    = nn.Linear(embed_dim, embed_dim, bias=False)
        self.o_proj    = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, embeddings, profile_mask):
        batch, N, D = embeddings.shape

        def split_heads(x):
            return x.reshape(batch, N, self.n_heads, self.head_dim).transpose(1, 2)

        Q = split_heads(self.q_proj(embeddings))
        K = split_heads(self.k_proj(embeddings))
        V = split_heads(self.v_proj(embeddings))

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if USE_SOFT_MASK:
            # Additive log-bias: log(0)=-inf fully excludes, log(0.5) halves influence
            soft_bias = torch.log(profile_mask.clamp(min=1e-9)).unsqueeze(1).unsqueeze(2)
            scores = scores + soft_bias
        else:
            # Hard exclusion: -inf before softmax → zero weight
            attn_mask = (profile_mask == 0).unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(attn_mask, float("-inf"))

        # nan_to_num handles the all-zero-mask edge case
        attn_weights = torch.nan_to_num(F_torch.softmax(scores, dim=-1), nan=0.0)

        out = torch.matmul(attn_weights, V).transpose(1, 2).reshape(batch, N, D)
        return embeddings + self.o_proj(out)   # residual connection


class AttentionExtractor(BaseFeaturesExtractor):
    """
    SB3 BaseFeaturesExtractor implementing the attention-based architecture.

    Observation layout [OBS_DIM = N*10 + N + 3]:
      [0      : N*10]      per-stock (9 features + current weight) × N stocks
      [N*10   : N*10+N]    profile mask values  (passed to attention gating)
      [N*10+N : N*10+N+3]  macro signals: egx30_ma_ratio, conia_delta, inflation_delta

    Processing:
      1. Reshape stock block → [batch, N, 10]
      2. SharedStockEncoder → [batch, N, 64]
      3. StockAttention (masked) → [batch, N, 64]
      4. Flatten → [batch, N*64]  then cat macro → [batch, N*64+3]

    Output [FEATURES_DIM = N*64 + 3] fed into SB3's actor and critic MLP heads.
    """
    def __init__(self, observation_space, n_stocks, encoder_input_dim,
                 encoder_hidden_dim, encoder_output_dim, n_heads):
        super().__init__(observation_space,
                         features_dim=n_stocks * encoder_output_dim + 3)
        self.n_stocks           = n_stocks
        self.encoder_input_dim  = encoder_input_dim
        self.encoder_output_dim = encoder_output_dim
        self.encoder   = SharedStockEncoder(encoder_input_dim,
                                            encoder_hidden_dim,
                                            encoder_output_dim)
        self.attention = StockAttention(encoder_output_dim, n_heads)

    def forward(self, obs):
        batch = obs.shape[0]
        N     = self.n_stocks
        F_in  = self.encoder_input_dim

        stock_flat    = obs[:, :N * F_in]
        profile_flat  = obs[:, N * F_in : N * F_in + N]
        macro_signals = obs[:, N * F_in + N : N * F_in + N + 3]

        embeddings = self.encoder(stock_flat.reshape(batch, N, F_in))
        attended   = self.attention(embeddings, profile_flat)

        return torch.cat(
            [attended.reshape(batch, N * self.encoder_output_dim), macro_signals],
            dim=1
        )


print(f"Architecture summary:")
print(f"  OBS_DIM      : {OBS_DIM}")
print(f"  FEATURES_DIM : {FEATURES_DIM}")
_dummy = spaces.Box(low=-np.inf, high=np.inf, shape=(OBS_DIM,), dtype=np.float32)
_ext   = AttentionExtractor(_dummy, N_STOCKS, ENCODER_INPUT_DIM,
                             ENCODER_HIDDEN_DIM, ENCODER_OUTPUT_DIM, ATTENTION_HEADS)
print(f"  Extractor params: {sum(p.numel() for p in _ext.parameters()):,}")
del _dummy, _ext

# %%

Architecture summary:
  OBS_DIM      : 375
  FEATURES_DIM : 1987
  Extractor params: 21,568


In [59]:
# =============================================================================
# CELL 12 — PortfolioEnv
# Fix 1 (_project_simplex): water-filling iterative cap. Never renormalizes
#   mid-loop. Residual after final hard clip absorbed into stocks (not just
#   cash) so sum=1 holds even when cash is at MAX_CASH_WEIGHT.
# Fix 2 (_apply_turnover_cap): explicit renorm after interpolation guarantees
#   sum=1 regardless of float32/float64 drift in current_weights.
# Fix 3 (_rolling_sharpe_reward order in step): rolling Sharpe computed
#   BEFORE appending current return to episode_returns.
# =============================================================================
class PortfolioEnv(gym.Env):
    def __init__(self, feature_tensor, returns_tensor, mask_tensor,
                 market_signal, profile_mask_tensor, universe_indices,
                 split_indices, risk_profile, feature_names,
                 mode="train", episode_length=24, rebalance_days=21,
                 silent=False, regime_median=1.0):   # ADD regime_median
        super().__init__()
        self.feature_tensor      = feature_tensor
        self.returns_tensor      = returns_tensor
        self.mask_tensor         = mask_tensor
        self.market_signal       = market_signal
        self.profile_mask_tensor = profile_mask_tensor
        self.universe_indices    = universe_indices
        self.feature_names       = feature_names
        self.risk_profile        = risk_profile
        self.mode                = mode
        self.episode_length      = episode_length
        self.rebalance_days      = rebalance_days
        self.regime_median       = regime_median     # ADD this line
        self.N           = len(universe_indices)
        self.F           = feature_tensor.shape[2]
        self.T           = feature_tensor.shape[0]
        self.reward_cfg  = REWARD_CONFIGS
        self.rel_vol_idx = REL_VOL_IDX

        rows_needed       = episode_length * rebalance_days
        self.valid_starts = [
            i for i in split_indices[mode] if i + rows_needed < self.T
        ]

        obs_dim = self.N * (self.F + 1) + self.N + 3
        assert obs_dim == OBS_DIM, (
            f"PortfolioEnv obs_dim={obs_dim} ≠ OBS_DIM={OBS_DIM}. "
            f"feature_tensor has {self.F} features; FEATURE_NAMES has "
            f"{len(feature_names)}. Rebuild tensors if FEATURE_NAMES changed."
        )

        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )
        self.action_space = spaces.Box(
            low=-10.0, high=10.0, shape=(self.N + 1,), dtype=np.float32
        )

        if not silent:
            print(f"PortfolioEnv [{mode}] : {self.N} stocks | "
                  f"{len(self.valid_starts)} valid starts | obs=({obs_dim},) | "
                  f"action={'delta' if USE_DELTA_ACTION else 'softmax'}")

        self.current_step    = 0
        self.start_idx       = 0
        self.current_weights = np.zeros(self.N + 1, dtype=np.float32)
        self.portfolio_value = 1.0
        self.peak_value      = 1.0
        self.episode_returns = []

    def _get_combined_mask(self, day_idx):
        avail   = self.mask_tensor[day_idx, self.universe_indices]
        profile = self.profile_mask_tensor[day_idx, self.universe_indices]
        if USE_SOFT_MASK:
            return (avail * profile).astype(np.float32)
        else:
            return (avail * (profile > 0).astype(float)).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if options is not None and "start_idx" in options:
            self.start_idx = options["start_idx"]
        elif self.mode == "train":
            self.start_idx = int(np.random.choice(self.valid_starts))
        else:
            self.start_idx = self.valid_starts[0]
        self.current_step    = 0
        self.portfolio_value = 1.0
        self.peak_value      = 1.0
        self.episode_returns = []
        self.current_weights = np.full(
            self.N + 1, 1.0 / (self.N + 1), dtype=np.float32
        )
        return self._get_observation(), {}

    def _get_observation(self):
        day_idx       = self.start_idx + self.current_step * self.rebalance_days
        combined_mask = self._get_combined_mask(day_idx)
        stock_features = self.feature_tensor[day_idx, self.universe_indices, :]
        stock_features = stock_features * (combined_mask > 0).astype(float)[:, np.newaxis]
        stock_with_weight = np.concatenate(
            [stock_features, self.current_weights[:-1].reshape(-1, 1)], axis=1
        ).astype(np.float32)
        return np.concatenate([
            stock_with_weight.flatten(),
            combined_mask,
            self.market_signal[day_idx].astype(np.float32),
        ]).astype(np.float32)

    def _action_to_weights(self, action, stock_mask):
        """Stage 1: convert raw PPO action to unconstrained proposed weights."""
        action = np.array(action, dtype=np.float32)
        if USE_DELTA_ACTION:
            stock_action = action[:-1]
            cash_action  = action[-1]
            eligible     = (stock_mask > 0).astype(float)
            effective_base = np.maximum(
                self.current_weights[:-1],
                MIN_ENTRY_FLOOR * eligible
            )
            stock_delta     = np.tanh(stock_action) * effective_base * DELTA_SCALE
            cash_base       = max(float(self.current_weights[-1]), MIN_ENTRY_FLOOR)
            cash_delta      = np.tanh(cash_action) * cash_base * DELTA_SCALE
            proposed_stocks = np.clip(self.current_weights[:-1] + stock_delta, 0.0, None)
            proposed_cash   = float(np.clip(self.current_weights[-1] + cash_delta, 0.0, None))
            proposed_stocks = proposed_stocks * eligible
            proposed        = np.append(proposed_stocks, proposed_cash).astype(np.float32)
            s = proposed.sum()
            if s > 1e-8:
                proposed = proposed / s
            else:
                n_eligible = int(eligible.sum())
                if n_eligible > 0:
                    proposed = np.append(
                        eligible / (n_eligible + 1),
                        [1.0 / (n_eligible + 1)]
                    ).astype(np.float32)
                else:
                    proposed = np.array([0.0] * self.N + [1.0], dtype=np.float32)
        else:
            exp_action = np.exp(action - action.max())
            proposed   = (exp_action / exp_action.sum()).astype(np.float32)
            proposed[:-1] *= stock_mask if USE_SOFT_MASK \
                             else (stock_mask > 0).astype(float)
            s = proposed.sum()
            proposed = proposed / s if s > 1e-8 else \
                       np.array([0.0] * self.N + [1.0], dtype=np.float32)
        return proposed

    def _project_simplex(self, weights, stock_mask):
        """
        Stage 2: water-filling iterative cap projection.

        Fix: added Step 10 — unconditional hard clip after residual absorption
        (Step 9). Proportional residual distribution in Step 9 can push stocks
        above MAX_STOCK_WEIGHT when the residual is non-trivial; without a
        re-clip the cap violation propagates to the return value.

        Fix: residual absorption now loops until convergence (max 3 passes)
        instead of a single proportional assignment, ensuring any cap violation
        introduced by the absorption is immediately clipped and redistributed.
        """
        w        = weights.copy().astype(np.float64)
        eligible = (stock_mask > 0).astype(bool)

        # Step 1: zero ineligible stocks
        w[:-1] = np.where(eligible, w[:-1], 0.0)

        # Step 2: clip cash
        w[-1] = float(np.clip(w[-1], 0.0, MAX_CASH_WEIGHT))

        # Step 3: clip stocks to per-stock cap
        w[:-1] = np.minimum(w[:-1], MAX_STOCK_WEIGHT)

        # Step 4: zero dust positions
        below_min = eligible & (w[:-1] < MIN_POSITION_SIZE)
        w[:-1]    = np.where(below_min, 0.0, w[:-1])

        # Step 5: first renorm → valid probability simplex
        s = w.sum()
        if s > 1e-8:
            w = w / s
        else:
            n_elig = int(eligible.sum())
            if n_elig > 0:
                w[:-1] = np.where(eligible, 1.0 / (n_elig + 1), 0.0)
                w[-1]  = 1.0 / (n_elig + 1)
            else:
                w     = np.zeros_like(w)
                w[-1] = 1.0

        # Step 6: water-filling loop — NO renorm inside.
        for _ in range(self.N + 2):
            over = w[:-1] > MAX_STOCK_WEIGHT + 1e-9
            if not over.any():
                break

            freed  = float((w[:-1][over] - MAX_STOCK_WEIGHT).sum())
            w[:-1] = np.minimum(w[:-1], MAX_STOCK_WEIGHT)

            cash_room = float(MAX_CASH_WEIGHT - w[-1])
            to_cash   = min(freed, max(cash_room, 0.0))
            w[-1]    += to_cash
            remaining = freed - to_cash

            if remaining > 1e-9:
                under_cap = eligible & (w[:-1] < MAX_STOCK_WEIGHT - 1e-9)
                if under_cap.any():
                    base     = np.where(under_cap, w[:-1], 0.0)
                    base_sum = base.sum()
                    if base_sum > 1e-8:
                        additions = remaining * (base / base_sum)
                    else:
                        additions = np.where(
                            under_cap,
                            remaining / float(under_cap.sum()),
                            0.0
                        )
                    w[:-1] = w[:-1] + additions
                else:
                    w[-1] = float(np.clip(w[-1] + remaining, 0.0, MAX_CASH_WEIGHT))

        # Step 7: single final renorm
        w[:-1] = np.where(eligible, w[:-1], 0.0)
        s = w.sum()
        if s > 1e-8:
            w = w / s
        else:
            w[-1] = 1.0

        # Step 8: unconditional hard clips to absorb float drift from renorm
        w[:-1] = np.minimum(w[:-1], MAX_STOCK_WEIGHT)
        w[-1]  = float(np.clip(w[-1], 0.0, MAX_CASH_WEIGHT))

        # Step 9 + 10: residual absorption loop (replaces single-pass proportional).
        # Each pass: compute residual → distribute proportionally to eligible stocks
        # under cap → hard-clip immediately → repeat until residual < 1e-10.
        # Looping prevents the single-pass version from introducing a new cap
        # violation that then goes uncorrected before return.
        for _ in range(5):
            residual = 1.0 - w.sum()
            if abs(residual) <= 1e-10:
                break

            under_cap = eligible & (w[:-1] < MAX_STOCK_WEIGHT - 1e-9)
            if under_cap.any():
                base     = np.where(under_cap, w[:-1], 0.0)
                base_sum = base.sum()
                if base_sum > 1e-8:
                    w[:-1] += residual * (base / base_sum)
                else:
                    w[:-1] = np.where(
                        under_cap,
                        w[:-1] + residual / float(under_cap.sum()),
                        w[:-1]
                    )
            else:
                w[-1] = float(np.clip(w[-1] + residual, 0.0, MAX_CASH_WEIGHT))

            # Hard clip after every absorption pass — this is the key fix.
            w[:-1] = np.minimum(w[:-1], MAX_STOCK_WEIGHT)
            w[-1]  = float(np.clip(w[-1], 0.0, MAX_CASH_WEIGHT))

        return w.astype(np.float32)

    def _apply_turnover_cap(self, new_weights):
        """
        Stage 3: scale trades proportionally if total turnover exceeds cap.

        Operates entirely in float64 and always renormalizes the result,
        preventing sum drift from float32 current_weights contaminating the
        interpolated portfolio.
        """
        cw = self.current_weights.astype(np.float64)
        nw = new_weights.astype(np.float64)

        proposed_turnover = float(np.sum(np.abs(nw - cw)))
        if proposed_turnover > TURNOVER_CAP:
            scale = TURNOVER_CAP / proposed_turnover
            nw    = cw + scale * (nw - cw)

        # Always renormalize — interpolation + float32 cast of cw can drift
        s = nw.sum()
        if s > 1e-8:
            nw = nw / s
        return nw.astype(np.float32)

    def _liquidity_adjusted_cost(self, day_idx, stock_turnover):
        rel_vol  = self.feature_tensor[
            day_idx, self.universe_indices, self.rel_vol_idx
        ]
        liq_mult = np.clip(
            1.3 - 0.3 * rel_vol,
            0.5,
            MAX_TRANSACTION_COST / BASE_TRANSACTION_COST
        )
        return float(np.dot(np.abs(stock_turnover), BASE_TRANSACTION_COST * liq_mult))

    def _rolling_sharpe_reward(self):
        """
        Called BEFORE appending current return to episode_returns.
        Returns 0.0 when fewer than 2 steps of history exist.
        """
        if SHARPE_REWARD_WINDOW == 0 or len(self.episode_returns) < 2:
            return 0.0
        window = np.array(
            self.episode_returns[-SHARPE_REWARD_WINDOW:], dtype=np.float64
        )
        mean_r = window.mean()
        std_r  = window.std()
        if std_r < 1e-8:
            return 0.0
        return float((mean_r - PERIOD_LOG_RFR) / std_r / np.sqrt(PERIODS_PER_YEAR))

    def step(self, action):
        day_idx    = self.start_idx + self.current_step * self.rebalance_days
        stock_mask = self._get_combined_mask(day_idx)

        raw_weights = self._action_to_weights(action, stock_mask)     # Stage 1
        new_weights = self._project_simplex(raw_weights, stock_mask)  # Stage 2
        new_weights = self._apply_turnover_cap(new_weights)           # Stage 3

        stock_turnover   = new_weights[:-1] - self.current_weights[:-1]
        turnover         = float(
            np.sum(np.abs(stock_turnover))
            + abs(new_weights[-1] - self.current_weights[-1])
        )
        transaction_cost = self._liquidity_adjusted_cost(day_idx, stock_turnover)

        eligible_mask  = (stock_mask > 0).astype(float)
        period_returns = self.returns_tensor[
            day_idx : day_idx + self.rebalance_days,
            self.universe_indices
        ]
        stock_ret_21d    = period_returns.sum(axis=0) * eligible_mask
        portfolio_return = float(np.dot(new_weights[:-1], stock_ret_21d))
        net_return       = portfolio_return - transaction_cost

        self.portfolio_value *= np.exp(net_return)
        self.peak_value       = max(self.peak_value, self.portfolio_value)

        # Compute rolling Sharpe BEFORE appending current return
        rolling_sharpe = self._rolling_sharpe_reward()
        self.episode_returns.append(net_return)

        drawdown = (self.peak_value - self.portfolio_value) / self.peak_value
        hhi      = float(np.sum(new_weights[:-1] ** 2))

        reward  = self.reward_cfg["return_weight"] * net_return
        reward += self.reward_cfg.get("sharpe_weight", 0.0) * rolling_sharpe
        reward -= self.reward_cfg["turnover_penalty"] * turnover
        if self.reward_cfg["downside_penalty"] > 0 and net_return < 0:
            reward -= self.reward_cfg["downside_penalty"] * abs(net_return)
        reward -= self.reward_cfg["drawdown_penalty"] * drawdown
        reward -= self.reward_cfg["hhi_penalty"] * hhi
        is_bullish = float(self.market_signal[day_idx, 0]) > self.regime_median
        if is_bullish and self.reward_cfg.get("cash_penalty", 0) > 0:
            reward -= self.reward_cfg["cash_penalty"] * \
                      max(0.0, new_weights[-1] - 0.10)

        self.current_weights = new_weights
        self.current_step   += 1
        truncated = self.current_step >= self.episode_length
        obs = self._get_observation() if not truncated else \
              np.zeros(self.observation_space.shape, dtype=np.float32)

        return obs, float(reward), False, truncated, {
            "portfolio_value":  self.portfolio_value,
            "portfolio_return": portfolio_return,
            "net_return":       net_return,
            "transaction_cost": transaction_cost,
            "turnover":         turnover,
            "drawdown":         drawdown,
            "hhi":              hhi,
            "rolling_sharpe":   rolling_sharpe,
            "current_weights":  new_weights.tolist(),
            "step":             self.current_step,
        }

    def close(self):
        pass

In [60]:
def build_fold_split_indices(fold):
    dates_pd   = pd.to_datetime(date_index)
    train_mask = dates_pd <= fold["train_end"]
    val_mask   = (dates_pd > fold["train_end"]) & (dates_pd <= fold["val_end"])
    test_mask  = (dates_pd > fold["val_end"])   & (dates_pd <= fold["test_end"])
    train_idx  = list(np.where(train_mask)[0])
    val_idx    = list(np.where(val_mask)[0])
    test_idx   = list(np.where(test_mask)[0])
    print(f"  {fold['name']}: train {date_index[train_idx[0]]}→{date_index[train_idx[-1]]} "
          f"({len(train_idx)}d)  val ({len(val_idx)}d)  test ({len(test_idx)}d)")
    return {"train": train_idx, "val": val_idx, "test": test_idx}

print("Walk-forward folds:")
fold_split_indices  = {}
fold_regime_medians = {}   # NEW

for fold in WALK_FORWARD_FOLDS:
    split = build_fold_split_indices(fold)
    fold_split_indices[fold["name"]] = split

    # Compute regime median on training indices only — no look-ahead bias
    train_signal = market_signal[split["train"], 0]
    median       = float(np.median(train_signal))
    fold_regime_medians[fold["name"]] = median
    bull_pct = (train_signal > median).mean() * 100
    print(f"    regime_median [{fold['name']}]: {median:.4f}  "
          f"(bull days in train: {bull_pct:.1f}%)")

_fold1_train_days = len(fold_split_indices["fold_1"]["train"])
print(f"\nTimestep scaling (base {BASE_TIMESTEPS:,} for fold_1):")
for fold in WALK_FORWARD_FOLDS:
    n  = len(fold_split_indices[fold["name"]]["train"])
    ts = int(BASE_TIMESTEPS * n / _fold1_train_days)
    print(f"  {fold['name']}: {n} days → {ts:,} timesteps")

print(f"\nPeriod log RFR : {PERIOD_LOG_RFR:.6f}  "
      f"(= ln(1+{ANNUAL_RISK_FREE_RATE:.4%}) / {PERIODS_PER_YEAR:.2f})")

Walk-forward folds:
  fold_1: train 2015-07-06→2019-12-31 (1092d)  val (243d)  test (244d)
    regime_median [fold_1]: 0.0471  (bull days in train: 50.0%)
  fold_2: train 2015-07-06→2020-12-31 (1335d)  val (244d)  test (244d)
    regime_median [fold_2]: 0.0354  (bull days in train: 50.0%)
  fold_3: train 2015-07-06→2021-12-30 (1579d)  val (244d)  test (242d)
    regime_median [fold_3]: 0.0673  (bull days in train: 50.0%)
  fold_4: train 2015-07-06→2022-12-29 (1823d)  val (242d)  test (484d)
    regime_median [fold_4]: 0.0780  (bull days in train: 50.0%)

Timestep scaling (base 3,500,000 for fold_1):
  fold_1: 1092 days → 3,500,000 timesteps
  fold_2: 1335 days → 4,278,846 timesteps
  fold_3: 1579 days → 5,060,897 timesteps
  fold_4: 1823 days → 5,842,948 timesteps

Period log RFR : 0.006569  (= ln(1+8.2024%) / 12.00)


In [61]:
class RewardLoggerCallback(BaseCallback):
    def __init__(self, log_path, verbose=0):
        super().__init__(verbose)
        self.log_path        = log_path
        self.episode_rewards = []
        self.episode_lengths = []
        self.episode_count   = 0

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_lengths.append(info["episode"]["l"])
                self.episode_count += 1
                if self.verbose > 0 and self.episode_count % 200 == 0:
                    recent = self.episode_rewards[-200:]
                    sys.__stdout__.write(
                        f"  Episode {self.episode_count:>6} | "
                        f"mean (last 200): {np.mean(recent):>8.4f}\n"
                    )
                    sys.__stdout__.flush()
        return True

    def _on_training_end(self):
        if not self.episode_rewards:
            return
        pd.DataFrame({
            "episode":        range(1, len(self.episode_rewards) + 1),
            "episode_reward": self.episode_rewards,
            "episode_length": self.episode_lengths,
        }).to_csv(self.log_path, index=False)
        sys.__stdout__.write(
            f"Training history saved : {self.log_path}\n"
            f"Total episodes         : {self.episode_count}\n"
            f"Final mean (last 200)  : "
            f"{np.mean(self.episode_rewards[-200:]):.4f}\n"
        )
        sys.__stdout__.flush()


class TrainingMetricsCallback(BaseCallback):
    def __init__(self, print_every_n_updates=20, verbose=1):
        super().__init__(verbose)
        self.print_every_n_updates = print_every_n_updates
        self.update_count          = 0

    def _on_rollout_end(self):
        self.update_count += 1
        if self.update_count % self.print_every_n_updates != 0:
            return True
        def gm(key):
            try:    return self.model.logger.name_to_value.get(key)
            except: return None
        parts = [f"  [{self.num_timesteps:>7,} steps | update {self.update_count:>4}]"]
        for k, label in [
            ("train/explained_variance", "expl_var"),
            ("train/clip_fraction",      "clip_frac"),
            ("train/value_loss",         "val_loss"),
        ]:
            v = gm(k)
            if v is not None:
                parts.append(f"{label}={v:>6.3f}")
        sys.__stdout__.write("  | ".join(parts) + "\n")
        sys.__stdout__.flush()
        return True

    def _on_step(self):
        return True


class EvalWithStopCallback(EvalCallback):
    def __init__(self, stop_callback, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.stop_callback = stop_callback

    def _on_step(self):
        cont = super()._on_step()
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            self.stop_callback.n_calls       = self.n_calls
            self.stop_callback.num_timesteps = self.num_timesteps
            self.stop_callback.parent        = self
            self.stop_callback.model         = self.model
            cont = cont and self.stop_callback._on_step()
        return cont

# %%

In [62]:

# =============================================================================
# CELL 15 — Helper functions
# =============================================================================
def compute_metrics(returns_arr, label, print_results=True):
    """
    Performance metrics from 21-day log returns.
 
    CAGR = exp(mean_log_return * periods_per_year) - 1
    Sharpe / Sortino: fully in log-return space.
    Max drawdown: reconstructed via exp(cumsum(log_returns)).
 
    Model assumption [1]: portfolio_return = dot(w, sum_21d_log_returns).
    Assumes constant intra-window weights. Realized CAGR in production
    may differ, particularly in high-volatility regimes.
    """
    if len(returns_arr) == 0:
        return {}
 
    ann_log_return = float(returns_arr.mean() * PERIODS_PER_YEAR)
    cagr           = float(np.exp(ann_log_return) - 1)
    ann_vol        = float(returns_arr.std() * np.sqrt(PERIODS_PER_YEAR))
    sharpe         = (ann_log_return - ANN_LOG_RFR) / (ann_vol + 1e-8)
 
    log_excess = returns_arr - PERIOD_LOG_RFR
    downside   = log_excess[log_excess < 0]
    sortino_v  = downside.std() * np.sqrt(PERIODS_PER_YEAR) if len(downside) > 0 else 1e-8
    sortino    = (ann_log_return - ANN_LOG_RFR) / (sortino_v + 1e-8)
 
    cum    = np.exp(np.cumsum(returns_arr))
    peak   = np.maximum.accumulate(cum)
    max_dd = float(((peak - cum) / peak).max())
    calmar = cagr / max_dd if max_dd > 0.001 else np.nan
 
    var_95   = np.percentile(returns_arr, 5)
    cvar_95  = float(returns_arr[returns_arr <= var_95].mean()) \
               if (returns_arr <= var_95).any() else np.nan
    win_rate = float((returns_arr > PERIOD_LOG_RFR).mean())
 
    if print_results:
        print(f"\n-- {label} --")
        print(f"  CAGR         : {cagr*100:>8.2f}%  |  Ann. Vol    : {ann_vol*100:>8.2f}%")
        print(f"  Sharpe       : {sharpe:>8.3f}  |  Sortino     : {sortino:>8.3f}")
        if not np.isnan(calmar):
            print(f"  Max Drawdown : {max_dd*100:>8.2f}%  |  Calmar      : {calmar:>8.3f}")
        else:
            print(f"  Max Drawdown : {max_dd*100:>8.2f}%  |  Calmar      :      N/A")
        cvar_str = f"{cvar_95*100:>7.2f}%" if not np.isnan(cvar_95) else "     N/A"
        print(f"  CVaR (95%)   : {cvar_str}  |  Win Rate RFR: {win_rate*100:>7.1f}%")
 
    return {
        "label": label, "n": len(returns_arr),
        "ann_return": cagr, "ann_vol": ann_vol,
        "sharpe": sharpe, "sortino": sortino,
        "max_dd": max_dd, "calmar": calmar,
        "cvar_95": cvar_95, "win_rate": win_rate,
    }
 
 
def moving_block_bootstrap(returns_arr, stat_fn,
                            n_bootstrap=N_BOOTSTRAP,
                            block_size=BOOTSTRAP_BLOCK,
                            ci=CONFIDENCE_LEVEL):
    """
    Moving-block bootstrap (non-circular).
 
    Replaces the previous np.roll circular bootstrap which wraps the
    end of the series onto the beginning, mixing distant regimes and
    destroying autocorrelation structure. Circular bootstrap produces
    confidence intervals that are too tight — a capital allocation
    red flag, not a cosmetic concern.
 
    This implementation samples only non-wrapping contiguous blocks
    so temporal structure and regime clustering within each block are
    preserved.
    """
    n         = len(returns_arr)
    max_start = n - block_size
    if max_start <= 0:
        return np.nan, np.nan
    n_blocks   = int(np.ceil(n / block_size))
    boot_stats = []
    for _ in range(n_bootstrap):
        starts    = np.random.randint(0, max_start + 1, size=n_blocks)
        resampled = np.concatenate(
            [returns_arr[s : s + block_size] for s in starts]
        )[:n]
        try:    boot_stats.append(stat_fn(resampled))
        except: continue
    boot_stats = np.array([s for s in boot_stats if np.isfinite(s)])
    if len(boot_stats) < 10:
        return np.nan, np.nan
    alpha = 1 - ci
    return (np.percentile(boot_stats, 100 * alpha / 2),
            np.percentile(boot_stats, 100 * (1 - alpha / 2)))
 
 
def compute_bootstrap_cis(returns_arr, label, print_results=True):
    """Moving-block bootstrap CIs for CAGR, Sharpe, and CVaR."""
    def sharpe_fn(r):
        ann_log = r.mean() * PERIODS_PER_YEAR
        ann_v   = r.std()  * np.sqrt(PERIODS_PER_YEAR)
        return (ann_log - ANN_LOG_RFR) / (ann_v + 1e-8)
 
    def ret_fn(r):
        return float(np.exp(r.mean() * PERIODS_PER_YEAR) - 1)
 
    def cvar_fn(r):
        v95  = np.percentile(r, 5)
        tail = r[r <= v95]
        return float(tail.mean()) if len(tail) > 0 else np.nan
 
    sharpe_lo, sharpe_hi = moving_block_bootstrap(returns_arr, sharpe_fn)
    ret_lo,    ret_hi    = moving_block_bootstrap(returns_arr, ret_fn)
    cvar_lo,   cvar_hi   = moving_block_bootstrap(returns_arr, cvar_fn)
 
    if print_results:
        print(f"\n  Bootstrap CIs ({int(CONFIDENCE_LEVEL*100)}%, MBB) — {label}")
        print(f"    CAGR       : [{ret_lo*100:+.2f}%, {ret_hi*100:+.2f}%]")
        print(f"    Sharpe     : [{sharpe_lo:+.3f}, {sharpe_hi:+.3f}]")
        print(f"    CVaR 95%   : [{cvar_lo*100:+.2f}%, {cvar_hi*100:+.2f}%]")
 
    return {
        "sharpe_ci": (sharpe_lo, sharpe_hi),
        "ret_ci":    (ret_lo, ret_hi),
        "cvar_ci":   (cvar_lo, cvar_hi),
    }
 
 
def significance_test(rl_returns, eq_returns, label=""):
    """
    HAC-corrected (Newey-West) t-test + Wilcoxon signed-rank test.
 
    Naive t-test assumes iid returns. Pooled evaluation returns from
    overlapping episodes are serially correlated, making naive p-values
    anti-conservative (appear more significant than they are).
    Newey-West correction accounts for autocorrelation up to HAC_LAGS lags.
 
    Note [2]: p-values measure difference in risk-adjusted returns including
    the RL cash timing option. Outperformance here ≠ pure selection alpha.
    Use no-cash ablation to isolate selection alpha.
    """
    n    = min(len(rl_returns), len(eq_returns))
    diff = (rl_returns[:n] - PERIOD_LOG_RFR) - (eq_returns[:n] - PERIOD_LOG_RFR)
 
    if _STATSMODELS_AVAILABLE:
        X       = add_constant(np.ones(n))
        result  = OLS(diff, X).fit(cov_type="HAC",
                                    cov_kwds={"maxlags": HAC_LAGS,
                                              "use_correction": True})
        t_stat  = float(result.tvalues[0])
        t_pval  = float(result.pvalues[0])
        t_label = f"HAC(lags={HAC_LAGS})"
    else:
        t_stat, t_pval = stats.ttest_1samp(diff, 0)
        t_label = "naive (install statsmodels for HAC)"
 
    try:    w_stat, w_pval = stats.wilcoxon(diff)
    except: w_stat, w_pval = np.nan, np.nan
 
    direction = "outperforms" if t_stat > 0 else "underperforms"
    print(f"  Significance ({label}): "
          f"t={t_stat:+.3f} p={t_pval:.4f} {'✓' if t_pval < 0.05 else '✗'} "
          f"[{t_label}] | "
          f"W p={w_pval:.4f} {'✓' if w_pval < 0.05 else '✗'} [{direction}]")
 
    return {
        "t_stat": t_stat, "t_pval": t_pval,
        "w_stat": w_stat, "w_pval": w_pval,
        "direction": direction, "t_type": t_label,
    }
 
 
def _make_eval_env(env_template, split_indices, mode, episode_length):
    return PortfolioEnv(
        feature_tensor      = env_template.feature_tensor,
        returns_tensor      = env_template.returns_tensor,
        mask_tensor         = env_template.mask_tensor,
        market_signal       = env_template.market_signal,
        profile_mask_tensor = env_template.profile_mask_tensor,
        universe_indices    = env_template.universe_indices,
        split_indices       = split_indices,
        risk_profile        = env_template.risk_profile,
        feature_names       = env_template.feature_names,
        mode                = mode,
        episode_length      = episode_length,
        rebalance_days      = env_template.rebalance_days,
        silent              = True,
        regime_median       = env_template.regime_median,   # ADD
    )
 
 
def _sample_valid_starts(eval_env, max_episodes):
    vs = eval_env.valid_starts
    if max_episodes and len(vs) > max_episodes:
        return [vs[i] for i in np.linspace(0, len(vs) - 1, max_episodes, dtype=int)]
    return vs
 
 
def multi_episode_evaluate(model, env_template, split_indices, mode,
                           episode_length, max_episodes=MAX_EVAL_EPISODES):
    eval_env     = _make_eval_env(env_template, split_indices, mode, episode_length)
    valid_starts = _sample_valid_starts(eval_env, max_episodes)
    all_returns, ep_metrics = [], []
 
    for start_idx in valid_starts:
        obs, _ = eval_env.reset(options={"start_idx": start_idx})
        ep_ret, ep_val = [], 1.0
        for _ in range(episode_length):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, _, truncated, info = eval_env.step(action)
            ep_ret.append(info["net_return"])
            ep_val = info["portfolio_value"]
            if truncated:
                break
        all_returns.extend(ep_ret)
        ep_metrics.append({
            "start_idx":   start_idx,
            "n_steps":     len(ep_ret),
            "final_value": ep_val,
            "mean_return": float(np.mean(ep_ret)),
        })
 
    return np.array(all_returns), pd.DataFrame(ep_metrics)
 
 
def multi_episode_evaluate_no_cash(model, env_template, split_indices, mode,
                                    episode_length, max_episodes=MAX_EVAL_EPISODES):
    """
    Cash-disabled ablation evaluation (model assumption [2]).

    Evaluates the existing policy with cash mechanically forced to zero
    after each step, redistributing all cash to stocks. Isolates stock
    selection alpha from the embedded cash timing option.

    Implementation: after each step, the cash weight is redistributed to
    stocks proportionally, then _project_simplex is called to enforce all
    stock constraints cleanly, then cash is zeroed and weights renormalized.
    This guarantees constraint compliance and sum=1 in all edge cases.

    If Sharpe drops materially vs standard evaluation, timing option value
    is a significant driver of reported alpha — this must be disclosed.

    Note: lower bound on a cash-free trained policy. A policy retrained
    with MAX_CASH_WEIGHT=0 would likely perform differently.
    """
    eval_env     = _make_eval_env(env_template, split_indices, mode, episode_length)
    valid_starts = _sample_valid_starts(eval_env, max_episodes)
    all_returns  = []

    for start_idx in valid_starts:
        obs, _ = eval_env.reset(options={"start_idx": start_idx})
        ep_ret = []
        for _ in range(episode_length):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, _, truncated, info = eval_env.step(action)

            # Step 1: read weights from environment step output
            weights   = np.array(info["current_weights"], dtype=np.float64)
            cash      = weights[-1]
            stock_sum = weights[:-1].sum()

            # Step 2: redistribute cash to stocks proportionally
            if cash > 1e-8 and stock_sum > 1e-8:
                weights[:-1] += cash * (weights[:-1] / stock_sum)
            weights[-1] = 0.0

            # Step 3: project through simplex to enforce per-stock cap cleanly
            # Use the mask at the current day to respect eligibility
            day_idx      = eval_env.start_idx + \
                           (eval_env.current_step - 1) * REBALANCE_DAYS
            stock_mask_nc = eval_env._get_combined_mask(day_idx)
            weights       = eval_env._project_simplex(
                weights.astype(np.float32), stock_mask_nc
            ).astype(np.float64)

            # Step 4: force cash to zero again (project_simplex may allow small
            # cash from the correction pass), redistribute remainder to stocks
            if weights[-1] > 1e-9:
                residual   = weights[-1]
                stock_sum2 = weights[:-1].sum()
                if stock_sum2 > 1e-8:
                    weights[:-1] += residual * (weights[:-1] / stock_sum2)
                weights[-1] = 0.0

            # Step 5: final renormalize
            s = weights.sum()
            if s > 1e-8:
                weights = weights / s

            # Recompute return using cash-free weights.
            # Transaction cost already paid on the original step — not recharged.
            eligible_mask = (stock_mask_nc > 0).astype(float)
            period_rets   = eval_env.returns_tensor[
                day_idx : day_idx + REBALANCE_DAYS,
                eval_env.universe_indices
            ]
            stock_ret_21d    = period_rets.sum(axis=0) * eligible_mask
            portfolio_return = float(np.dot(weights[:-1], stock_ret_21d))
            net_return       = portfolio_return - info["transaction_cost"]
            ep_ret.append(net_return)

            if truncated:
                break
        all_returns.extend(ep_ret)

    return np.array(all_returns)

 
 
def multi_episode_equal_weight(env_template, split_indices, mode,
                               episode_length, max_episodes=MAX_EVAL_EPISODES):
    """
    Equal-weight baseline — always fully invested, zero cash.
 
    Note (model assumption [2]): RL agent can hold up to 30% cash,
    giving it an embedded timing option absent from this benchmark.
    Outperformance on Sharpe vs EQW reflects both selection alpha and
    timing option value. Use multi_episode_evaluate_no_cash() alongside
    this to decompose the two effects.
    """
    eval_env     = _make_eval_env(env_template, split_indices, mode, episode_length)
    valid_starts = _sample_valid_starts(eval_env, max_episodes)
    all_returns  = []
    N_full       = len(env_template.universe_indices) + 1
 
    for start_idx in valid_starts:
        eq_prev_full = np.full(N_full, 1.0 / N_full, dtype=np.float32)
        ep_ret       = []
 
        for step in range(episode_length):
            day_idx = start_idx + step * env_template.rebalance_days
            if day_idx + env_template.rebalance_days >= env_template.T:
                break
 
            stock_mask = eval_env._get_combined_mask(day_idx)
            n_valid    = (stock_mask > 0).sum()
            eq_w       = (stock_mask > 0).astype(float) / n_valid \
                         if n_valid > 0 \
                         else np.zeros(len(env_template.universe_indices))
            eq_full    = np.concatenate([eq_w, [0.0]])
 
            stock_delta      = eq_w - eq_prev_full[:-1]
            transaction_cost = eval_env._liquidity_adjusted_cost(day_idx, stock_delta)
            eq_prev_full     = eq_full.copy()
 
            period_rets = env_template.returns_tensor[
                day_idx : day_idx + env_template.rebalance_days,
                env_template.universe_indices
            ]
            gross = float(np.dot(
                eq_w,
                period_rets.sum(axis=0) * (stock_mask > 0).astype(float)
            ))
            ep_ret.append(gross - transaction_cost)
 
        all_returns.extend(ep_ret)
 
    return np.array(all_returns)
 
 
def compute_active_metrics(rl_returns, eq_returns, model, env_template,
                           split_indices, mode, episode_length, fold_name,
                           max_episodes=MAX_EVAL_EPISODES, print_results=True):
    """
    Active metrics: pooled IR, win rate, turnover, active stocks.
    IR is pooled across overlapping episodes — inflated by serial correlation.
    Prefer compute_ir_single_pass() for thesis reporting.
    """
    n        = min(len(rl_returns), len(eq_returns))
    active   = rl_returns[:n] - eq_returns[:n]
    ir       = float(active.mean() / (active.std() + 1e-8)) * np.sqrt(PERIODS_PER_YEAR)
    wr_vs_eq = float((rl_returns[:n] > eq_returns[:n]).mean())
 
    eval_env     = _make_eval_env(env_template, split_indices, mode, episode_length)
    valid_starts = _sample_valid_starts(eval_env, max_episodes)
    all_turnovers, all_active_stocks = [], []
 
    for start_idx in valid_starts:
        obs, _ = eval_env.reset(options={"start_idx": start_idx})
        for _ in range(episode_length):
            day_idx    = eval_env.start_idx + eval_env.current_step * REBALANCE_DAYS
            stock_mask = eval_env._get_combined_mask(day_idx)
            all_active_stocks.append(int((stock_mask > 0).sum()))
            action, _ = model.predict(obs, deterministic=True)
            obs, _, _, truncated, info = eval_env.step(action)
            all_turnovers.append(info["turnover"])
            if truncated:
                break
 
    avg_turnover = float(np.mean(all_turnovers))
    avg_stocks   = float(np.mean(all_active_stocks))
 
    if print_results:
        print(f"\n-- Active Metrics [{fold_name} {mode}] --")
        print(f"  Information Ratio : {ir:>8.3f}  (pooled; prefer single-pass IR)")
        print(f"  Win Rate vs EQW   : {wr_vs_eq*100:>7.1f}%")
        print(f"  Avg Turnover/step : {avg_turnover*100:>7.1f}%  "
              f"(full round-trip; ÷2 for academic one-way)")
        print(f"  Avg Active Stocks : {avg_stocks:>7.1f}")
 
    return {
        "information_ratio": ir,
        "win_rate_vs_eq":    wr_vs_eq,
        "avg_turnover":      avg_turnover,
        "avg_active_stocks": avg_stocks,
    }
 
 
def compute_ir_single_pass(model, env_template, split_indices,
                           episode_length, fold_name, print_results=True):
    """
    Single-pass IR over one non-overlapping episode through the test period.
    Primary reported IR — avoids serial correlation from pooled episodes.
    """
    eval_env  = _make_eval_env(env_template, split_indices, "test", episode_length)
    start_idx = eval_env.valid_starts[0]
    N_full    = len(env_template.universe_indices) + 1
 
    rl_rets, eq_rets = [], []
    obs, _ = eval_env.reset(options={"start_idx": start_idx})
    eq_prev_full = np.full(N_full, 1.0 / N_full, dtype=np.float32)
 
    for step in range(episode_length):
        day_idx    = eval_env.start_idx + eval_env.current_step * REBALANCE_DAYS
        stock_mask = eval_env._get_combined_mask(day_idx)
        action, _  = model.predict(obs, deterministic=True)
        obs, _, _, truncated, info = eval_env.step(action)
        rl_rets.append(info["net_return"])
 
        n_valid  = (stock_mask > 0).sum()
        eq_w     = (stock_mask > 0).astype(float) / n_valid if n_valid > 0 \
                   else np.zeros(eval_env.N)
        eq_full  = np.concatenate([eq_w, [0.0]])
        eq_cost  = eval_env._liquidity_adjusted_cost(
            day_idx, eq_w - eq_prev_full[:-1]
        )
        eq_prev_full = eq_full.copy()
 
        period_rets = env_template.returns_tensor[
            day_idx : day_idx + REBALANCE_DAYS,
            env_template.universe_indices
        ]
        gross = float(np.dot(
            eq_w,
            period_rets.sum(axis=0) * (stock_mask > 0).astype(float)
        ))
        eq_rets.append(gross - eq_cost)
 
        if truncated:
            break
 
    rl_arr = np.array(rl_rets)
    eq_arr = np.array(eq_rets[:len(rl_arr)])
    active = rl_arr - eq_arr
    ir     = float(active.mean() / (active.std() + 1e-8)) * np.sqrt(PERIODS_PER_YEAR)
 
    if print_results:
        print(f"  IR single-pass [{fold_name}]: {ir:>8.3f}  "
              f"(non-overlapping, n={len(active)} periods)")
 
    return ir
 
 
def extract_portfolio_allocations(model, env_template, split_indices,
                                  mode="test", episode_length=EPISODE_LENGTH,
                                  max_episodes=MAX_EVAL_EPISODES):
    """Record every portfolio rebalance allocation across evaluation episodes."""
    eval_env     = _make_eval_env(env_template, split_indices, mode, episode_length)
    valid_starts = _sample_valid_starts(eval_env, max_episodes)
    rows         = []
 
    for ep_num, start_idx in enumerate(valid_starts):
        obs, _ = eval_env.reset(options={"start_idx": start_idx})
        for rebalance_num in range(episode_length):
            day_idx = eval_env.start_idx + eval_env.current_step * REBALANCE_DAYS
            action, _ = model.predict(obs, deterministic=True)
            obs, _, _, truncated, info = eval_env.step(action)
            row = {
                "episode":   ep_num,
                "rebalance": rebalance_num,
                "date":      date_index[day_idx],
                "cash":      info["current_weights"][-1],
            }
            for local_idx, global_idx in enumerate(universe_indices):
                row[tickers[global_idx]] = info["current_weights"][local_idx]
            rows.append(row)
            if truncated:
                break
 
    return pd.DataFrame(rows)

In [63]:

# =============================================================================
# SANITY CHECK CELL
# Fix (Issue 4): tightened stock cap tolerance from 1e-5 to 1e-6 to reflect
#   the float32 output of _project_simplex (max drift ~1e-7).
#   Also added explicit rolling_sharpe diagnostic to verify Issue 2 fix:
#   rolling_sharpe at step 0 must be 0.0 (no history yet).
# =============================================================================
_sanity_split = fold_split_indices["fold_4"]
sanity_env = PortfolioEnv(
    feature_tensor      = feature_tensor,
    returns_tensor      = returns_tensor,
    mask_tensor         = mask_tensor,
    market_signal       = market_signal,
    profile_mask_tensor = profile_mask_tensor,
    universe_indices    = universe_indices,
    split_indices       = _sanity_split,
    risk_profile        = RISK_PROFILE,
    feature_names       = FEATURE_NAMES,
    mode                = "train",
    episode_length      = EPISODE_LENGTH,
    rebalance_days      = REBALANCE_DAYS,
    regime_median       = fold_regime_medians["fold_4"],   # ADD
)

obs, _ = sanity_env.reset(seed=42)
expected_obs_dim = N_STOCKS * (len(FEATURE_NAMES) + 1) + N_STOCKS + 3

assert obs.shape[0] == expected_obs_dim == OBS_DIM, \
    f"Obs dim mismatch: {obs.shape[0]} vs {expected_obs_dim}"
assert not np.isnan(obs).any(), "NaN in observation"

macro_in_obs = obs[-3:]
assert np.allclose(macro_in_obs, market_signal[sanity_env.start_idx], atol=1e-5), \
    "Macro signal mismatch"

profile_start  = N_STOCKS * (len(FEATURE_NAMES) + 1)
profile_in_obs = obs[profile_start : profile_start + N_STOCKS]
day_mask       = sanity_env._get_combined_mask(sanity_env.start_idx)
assert np.allclose(profile_in_obs, day_mask, atol=1e-5), \
    "Profile mask mismatch in observation"

rolling_sharpe_step0 = None
for step in range(EPISODE_LENGTH):
    action = np.random.randn(sanity_env.action_space.shape[0]).astype(np.float32)
    obs, reward, _, truncated, info = sanity_env.step(action)

    weights       = np.array(info["current_weights"])
    stock_weights = weights[:-1]

    # Fix (Issue 4): tightened tolerance — _project_simplex returns float32
    # with max drift ~1e-7; 1e-5 was too permissive for post-deployment use.
    assert not np.isnan(reward), \
        f"NaN reward at step {step}"
    assert np.isfinite(weights).all(), \
        f"Non-finite weights at step {step}"
    assert np.all(stock_weights <= MAX_STOCK_WEIGHT + 1e-6), \
        f"Stock cap violated: max={stock_weights.max():.8f} at step {step}"
    assert weights[-1] <= MAX_CASH_WEIGHT + 1e-6, \
        f"Cash cap violated: {weights[-1]:.8f} at step {step}"
    assert abs(weights.sum() - 1.0) < 1e-5, \
        f"Weights sum = {weights.sum():.8f} at step {step}"

    # Fix (Issue 2) verification: at step 0 rolling_sharpe must be 0.0
    # because episode_returns was empty when _rolling_sharpe_reward was called.
    if step == 0:
        rolling_sharpe_step0 = info["rolling_sharpe"]
        assert rolling_sharpe_step0 == 0.0, (
            f"rolling_sharpe at step 0 should be 0.0 (no history), "
            f"got {rolling_sharpe_step0}. Double-count fix may be broken."
        )

    if truncated:
        break

_obs_space = spaces.Box(low=-np.inf, high=np.inf, shape=(OBS_DIM,), dtype=np.float32)
_ext = AttentionExtractor(
    _obs_space, N_STOCKS, ENCODER_INPUT_DIM,
    ENCODER_HIDDEN_DIM, ENCODER_OUTPUT_DIM, ATTENTION_HEADS
)
with torch.no_grad():
    _out = _ext(torch.from_numpy(obs).unsqueeze(0))
assert _out.shape == (1, FEATURES_DIM) and torch.isfinite(_out).all()
del _obs_space, _ext, _out

print(f"✅ Sanity check passed.")
print(f"  obs_dim              : {obs.shape[0]} ✓")
print(f"  Active stocks        : {int((day_mask > 0).sum())}/{N_STOCKS}")
print(f"  Episode steps        : {step+1}/{EPISODE_LENGTH} ✓")
print(f"  Extractor out        : (1, {FEATURES_DIM}) ✓")
print(f"  Max stock wt         : {stock_weights.max():.8f} ≤ {MAX_STOCK_WEIGHT} ✓")
print(f"  Cash weight          : {weights[-1]:.8f} ≤ {MAX_CASH_WEIGHT} ✓")
print(f"  Weights sum          : {weights.sum():.8f} ✓")
print(f"  rolling_sharpe step0 : {rolling_sharpe_step0} == 0.0 ✓  (no double-count)")

PortfolioEnv [train] : 31 stocks | 1823 valid starts | obs=(375,) | action=delta
✅ Sanity check passed.
  obs_dim              : 375 ✓
  Active stocks        : 9/31
  Episode steps        : 24/24 ✓
  Extractor out        : (1, 1987) ✓
  Max stock wt         : 0.19914739 ≤ 0.2 ✓
  Cash weight          : 0.30000001 ≤ 0.3 ✓
  Weights sum          : 1.00000000 ✓
  rolling_sharpe step0 : 0.0 == 0.0 ✓  (no double-count)


In [ ]:
fold_models = {}

_template_env = PortfolioEnv(
    feature_tensor      = feature_tensor,
    returns_tensor      = returns_tensor,
    mask_tensor         = mask_tensor,
    market_signal       = market_signal,
    profile_mask_tensor = profile_mask_tensor,
    universe_indices    = universe_indices,
    split_indices       = fold_split_indices["fold_4"],
    risk_profile        = RISK_PROFILE,
    feature_names       = FEATURE_NAMES,
    mode                = "test",
    episode_length      = EPISODE_LENGTH,
    rebalance_days      = REBALANCE_DAYS,
    silent              = True,
    regime_median       = fold_regime_medians["fold_4"],   # ADD
)

for fold in WALK_FORWARD_FOLDS:
    # Remove or change this line to train different folds
    if fold["name"] != "fold_4":
        continue

    fold_name      = fold["name"]
    split          = fold_split_indices[fold_name]
    n_train_days   = len(split["train"])
    fold_timesteps = int(BASE_TIMESTEPS * n_train_days / _fold1_train_days)

    fold_dir          = VERSION_DIR / fold_name
    fold_dir.mkdir(parents=True, exist_ok=True)
    model_staging_dir = fold_dir / "_staging"
    model_staging_dir.mkdir(parents=True, exist_ok=True)
    final_model_path  = fold_dir / f"{VERSION}_{fold_name}.zip"

    print(f"\n{'='*62}")
    print(f"  TRAINING — {fold_name}  (→val {fold['val_end']})")
    print(f"  Train days : {n_train_days}  |  Timesteps : {fold_timesteps:,}")
    print(f"  Output dir : {fold_dir}")
    print(f"{'='*62}")

    # Update template env regime_median for this fold
    # (used by _make_eval_env in evaluation functions)
    _template_env.regime_median = fold_regime_medians[fold_name]   # ADD

    def make_env_fn(s=split, median=fold_regime_medians[fold_name]):   # ADD median arg
        def _init():
            return PortfolioEnv(
                feature_tensor      = feature_tensor,
                returns_tensor      = returns_tensor,
                mask_tensor         = mask_tensor,
                market_signal       = market_signal,
                profile_mask_tensor = profile_mask_tensor,
                universe_indices    = universe_indices,
                split_indices       = s,
                risk_profile        = RISK_PROFILE,
                feature_names       = FEATURE_NAMES,
                mode                = "train",
                episode_length      = EPISODE_LENGTH,
                rebalance_days      = REBALANCE_DAYS,
                silent              = True,
                regime_median       = median,   # ADD
            )
        return _init

    val_env_monitored = Monitor(PortfolioEnv(
        feature_tensor      = feature_tensor,
        returns_tensor      = returns_tensor,
        mask_tensor         = mask_tensor,
        market_signal       = market_signal,
        profile_mask_tensor = profile_mask_tensor,
        universe_indices    = universe_indices,
        split_indices       = split,
        risk_profile        = RISK_PROFILE,
        feature_names       = FEATURE_NAMES,
        mode                = "val",
        episode_length      = EPISODE_LENGTH,
        rebalance_days      = REBALANCE_DAYS,
        silent              = True,
        regime_median       = fold_regime_medians[fold_name],   # ADD
    ))

    policy_kwargs = dict(
        features_extractor_class  = AttentionExtractor,
        features_extractor_kwargs = dict(
            n_stocks           = N_STOCKS,
            encoder_input_dim  = ENCODER_INPUT_DIM,
            encoder_hidden_dim = ENCODER_HIDDEN_DIM,
            encoder_output_dim = ENCODER_OUTPUT_DIM,
            n_heads            = ATTENTION_HEADS,
        ),
        net_arch      = POLICY_HIDDEN_DIMS,
        activation_fn = nn.Tanh,
    )

    train_vec = VecMonitor(DummyVecEnv([make_env_fn(split) for _ in range(N_ENVS)]))

    model = PPO(
        policy        = "MlpPolicy",
        env           = train_vec,
        seed          = 42,
        policy_kwargs = policy_kwargs,
        **PPO_PARAMS,
    )
    print(f"  Parameters : {sum(p.numel() for p in model.policy.parameters()):,}")

    stop_callback = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals = 50,
        min_evals                = 60,
        verbose                  = 1,
    )
    eval_callback = EvalWithStopCallback(
        stop_callback        = stop_callback,
        eval_env             = val_env_monitored,
        best_model_save_path = str(model_staging_dir),
        log_path             = str(fold_dir / "evaluations"),   # → evaluations.npz in fold_dir
        eval_freq            = max(20_000 // N_ENVS, 1),
        n_eval_episodes      = 1,
        deterministic        = True,
        render               = False,
        verbose              = 1,
    )
    reward_logger = RewardLoggerCallback(
        log_path = str(fold_dir / "training_history.csv"),
        verbose  = 1,
    )
    metrics_cb = TrainingMetricsCallback(print_every_n_updates=20, verbose=1)

    model.learn(
        total_timesteps     = fold_timesteps,
        callback            = [eval_callback, reward_logger, metrics_cb],
        progress_bar        = False,
        reset_num_timesteps = True,
    )

    # Rename staged model to versioned filename
    staged_model = model_staging_dir / "best_model.zip"
    if staged_model.exists():
        staged_model.rename(final_model_path)
        print(f"  Model saved : {final_model_path.name}")
    else:
        model.save(str(final_model_path.with_suffix("")))
        print(f"  Model saved (fallback) : {final_model_path.name}")

    try:
        model_staging_dir.rmdir()
    except OSError:
        pass

    best_model = PPO.load(str(final_model_path), env=val_env_monitored)
    fold_models[fold_name] = best_model
    print(f"  {fold_name} complete.")

# %%


  TRAINING — fold_4  (→val 2023-12-31)
  Train days : 1823  |  Timesteps : 5,842,948
  Output dir : C:\Users\mirae\Desktop\Personalization_Engine\model_versions\v7_higher_cash_penalty\fold_4
  Parameters : 1,179,521
Eval num_timesteps=20000, episode_reward=-16.26 +/- 0.00
Episode length: 24.00 +/- 0.00
New best mean reward!
Eval num_timesteps=40000, episode_reward=-16.11 +/- 0.00
Episode length: 24.00 +/- 0.00
New best mean reward!
Eval num_timesteps=60000, episode_reward=-15.83 +/- 0.00
Episode length: 24.00 +/- 0.00
New best mean reward!
Eval num_timesteps=80000, episode_reward=-14.87 +/- 0.00
Episode length: 24.00 +/- 0.00
New best mean reward!
Eval num_timesteps=100000, episode_reward=-15.65 +/- 0.00
Episode length: 24.00 +/- 0.00
Eval num_timesteps=120000, episode_reward=-16.71 +/- 0.00
Episode length: 24.00 +/- 0.00
Eval num_timesteps=140000, episode_reward=-15.64 +/- 0.00
Episode length: 24.00 +/- 0.00
Eval num_timesteps=160000, episode_reward=-16.24 +/- 0.00
Episode length: 24

In [ ]:
all_fold_results = {}

for fold_name, model in fold_models.items():
    fold  = next(f for f in WALK_FORWARD_FOLDS if f["name"] == fold_name)
    split = fold_split_indices[fold_name]

    print(f"\n{'='*60}")
    print(f"  EVALUATION — {fold_name}  (test → {fold['test_end']})")
    print(f"{'='*60}")

    # --- Standard evaluation ---
    rl_val_returns,  _ = multi_episode_evaluate(
        model, _template_env, split, "val",  EPISODE_LENGTH
    )
    eq_val_returns     = multi_episode_equal_weight(
        _template_env, split, "val",  EPISODE_LENGTH
    )
    rl_test_returns, _ = multi_episode_evaluate(
        model, _template_env, split, "test", EPISODE_LENGTH
    )
    eq_test_returns    = multi_episode_equal_weight(
        _template_env, split, "test", EPISODE_LENGTH
    )

    rl_val_metrics  = compute_metrics(rl_val_returns,  f"RL Agent — val [{fold_name}]")
    eq_val_metrics  = compute_metrics(eq_val_returns,  f"Equal Weight — val [{fold_name}]")
    rl_val_ci       = compute_bootstrap_cis(rl_val_returns,  f"RL val [{fold_name}]")
    val_sig         = significance_test(
        rl_val_returns[:min(len(rl_val_returns), len(eq_val_returns))],
        eq_val_returns[:min(len(rl_val_returns), len(eq_val_returns))],
        label=f"val {fold_name}",
    )

    rl_test_metrics = compute_metrics(rl_test_returns, f"RL Agent — test [{fold_name}]")
    eq_test_metrics = compute_metrics(eq_test_returns, f"Equal Weight — test [{fold_name}]")
    rl_test_ci      = compute_bootstrap_cis(rl_test_returns, f"RL test [{fold_name}]")
    test_sig        = significance_test(
        rl_test_returns[:min(len(rl_test_returns), len(eq_test_returns))],
        eq_test_returns[:min(len(rl_test_returns), len(eq_test_returns))],
        label=f"test {fold_name}",
    )

    active_metrics = compute_active_metrics(
        rl_returns     = rl_test_returns,
        eq_returns     = eq_test_returns,
        model          = model,
        env_template   = _template_env,
        split_indices  = split,
        mode           = "test",
        episode_length = EPISODE_LENGTH,
        fold_name      = fold_name,
    )

    # --- Single-pass IR (primary reported IR — no serial correlation) ---
    ir_single = compute_ir_single_pass(
        model, _template_env, split, EPISODE_LENGTH, fold_name
    )
    active_metrics["information_ratio_single_pass"] = ir_single

    # --- Cash timing option ablation (model assumption [2]) ---
    # Redistributes cash to stocks after each step to isolate selection alpha.
    # Sharpe drop vs standard evaluation = timing option contribution.
    print(f"\n-- Cash Timing Option Ablation [{fold_name} test] --")
    print(f"  (Removes RL cash allocation to isolate stock selection alpha)")
    rl_test_no_cash = multi_episode_evaluate_no_cash(
        model, _template_env, split, "test", EPISODE_LENGTH
    )
    no_cash_metrics = compute_metrics(
        rl_test_no_cash,
        f"RL Agent (no cash) — test [{fold_name}]"
    )
    sharpe_drop = rl_test_metrics["sharpe"] - no_cash_metrics["sharpe"]
    print(f"  Sharpe with cash    : {rl_test_metrics['sharpe']:+.3f}")
    print(f"  Sharpe without cash : {no_cash_metrics['sharpe']:+.3f}")
    print(f"  Timing option value : {sharpe_drop:+.3f} Sharpe units "
          f"({'significant' if abs(sharpe_drop) > 0.1 else 'modest'})")

    # --- Portfolio allocations ---
    allocations_df   = extract_portfolio_allocations(
        model, _template_env, split, mode="test"
    )
    allocations_path = VERSION_DIR / fold_name / "portfolio_allocations.csv"
    allocations_df.to_csv(allocations_path, index=False)
    print(f"\n  Allocations saved : {allocations_path}")

    fold_result = {
        "fold":              fold_name,
        "version":           VERSION,
        "train_end":         fold["train_end"],
        "val_end":           fold["val_end"],
        "test_end":          fold["test_end"],
        "model_file":        f"{VERSION}_{fold_name}.zip",
        "rl_val":            rl_val_metrics,
        "eq_val":            eq_val_metrics,
        "rl_val_ci":         rl_val_ci,
        "rl_test":           rl_test_metrics,
        "eq_test":           eq_test_metrics,
        "rl_test_ci":        rl_test_ci,
        "val_significance":  val_sig,
        "test_significance": test_sig,
        "active_metrics":    active_metrics,
        # Cash timing option ablation — model assumption [2]
        "rl_test_no_cash":         no_cash_metrics,
        "timing_option_sharpe":    float(sharpe_drop),
    }
    all_fold_results[fold_name] = fold_result

    with open(VERSION_DIR / fold_name / "metrics.json", "w") as f:
        json.dump(fold_result, f, indent=2, default=str)
    print(f"  Metrics saved     : {VERSION_DIR / fold_name / 'metrics.json'}")

print(f"\n✅ All folds evaluated.")


  EVALUATION — fold_4  (test → 2025-12-31)



-- RL Agent — val [fold_4] --
  CAGR         :    28.82%  |  Ann. Vol    :    17.07%
  Sharpe       :    1.021  |  Sortino     :    1.915
  Max Drawdown :    16.41%  |  Calmar      :    1.756
  CVaR (95%)   :   -7.62%  |  Win Rate RFR:    62.3%

-- Equal Weight — val [fold_4] --
  CAGR         :    30.55%  |  Ann. Vol    :    25.36%
  Sharpe       :    0.740  |  Sortino     :    0.945
  Max Drawdown :    28.39%  |  Calmar      :    1.076
  CVaR (95%)   :  -16.54%  |  Win Rate RFR:    63.7%

  Bootstrap CIs (95%, MBB) — RL val [fold_4]
    CAGR       : [+24.81%, +32.94%]
    Sharpe     : [+0.833, +1.208]
    CVaR 95%   : [-8.27%, -6.95%]
  Significance (val fold_4): t=-0.825 p=0.4095 ✗ [HAC(lags=3)] | W p=0.0000 ✓ [underperforms]

-- RL Agent — test [fold_4] --
  CAGR         :    17.17%  |  Ann. Vol    :    14.56%
  Sharpe       :    0.547  |  Sortino     :    0.790
  Max Drawdown :    27.89%  |  Calmar      :    0.616
  CVaR (95%)   :   -8.74%  |  Win Rate RFR:    60.1%

-- Equal Wei

In [ ]:
n_trained = len(all_fold_results)

print(f"\n{'='*65}")
print(f"  CROSS-FOLD AGGREGATE — {VERSION}")
print(f"  {n_trained}/{len(WALK_FORWARD_FOLDS)} folds trained")
print(f"{'='*65}")

if n_trained < 2:
    print(f"\n  ⚠ WARNING: Only {n_trained} fold(s) trained.")
    print(f"    Std = 0.000 across all metrics — not a meaningful cross-fold result.")
    print(f"    Run all 4 folds before reporting aggregate statistics.")
elif n_trained < len(WALK_FORWARD_FOLDS):
    print(f"\n  ⚠ NOTE: {len(WALK_FORWARD_FOLDS) - n_trained} fold(s) not yet trained.")

metrics_keys = ["ann_return", "ann_vol", "sharpe", "sortino", "max_dd", "calmar", "cvar_95"]
rl_sharpes   = [all_fold_results[f]["rl_test"]["sharpe"]         for f in all_fold_results]
eq_sharpes   = [all_fold_results[f]["eq_test"]["sharpe"]         for f in all_fold_results]
rl_returns   = [all_fold_results[f]["rl_test"]["ann_return"]*100 for f in all_fold_results]
eq_returns   = [all_fold_results[f]["eq_test"]["ann_return"]*100 for f in all_fold_results]
nc_sharpes   = [all_fold_results[f]["rl_test_no_cash"]["sharpe"] for f in all_fold_results]
timing_vals  = [all_fold_results[f]["timing_option_sharpe"]      for f in all_fold_results]

print(f"\n{'Metric':<22} {'RL Mean':>10} {'RL Std':>8} | {'EQ Mean':>10} {'EQ Std':>8}")
print("-" * 67)
for k in metrics_keys:
    rl_vals = [all_fold_results[f]["rl_test"].get(k, np.nan) for f in all_fold_results]
    eq_vals = [all_fold_results[f]["eq_test"].get(k, np.nan) for f in all_fold_results]
    scale   = 100 if k in ["ann_return", "ann_vol", "max_dd", "cvar_95"] else 1
    unit    = "%" if scale == 100 else ""
    label   = "CAGR" if k == "ann_return" else k
    print(f"{label:<22} {np.nanmean(rl_vals)*scale:>9.2f}{unit} "
          f"{np.nanstd(rl_vals)*scale:>7.2f}{unit} | "
          f"{np.nanmean(eq_vals)*scale:>9.2f}{unit} "
          f"{np.nanstd(eq_vals)*scale:>7.2f}{unit}")

print(f"\n-- Cash Timing Option Summary (model assumption [2]) --")
print(f"  RL mean Sharpe (with cash)    : {np.mean(rl_sharpes):+.3f}")
print(f"  RL mean Sharpe (no cash)      : {np.mean(nc_sharpes):+.3f}")
print(f"  Mean timing option value      : {np.mean(timing_vals):+.3f} Sharpe units")
print(f"  (Positive = cash allocation helped; isolates selection alpha)")

n_out = sum(
    1 for f in all_fold_results
    if all_fold_results[f]["test_significance"].get("t_pval", 1) < 0.05
    and all_fold_results[f]["test_significance"].get("direction") == "outperforms"
)
n_under = sum(
    1 for f in all_fold_results
    if all_fold_results[f]["test_significance"].get("t_pval", 1) < 0.05
    and all_fold_results[f]["test_significance"].get("direction") == "underperforms"
)
print(f"\nSignificantly outperforms EQW   : {n_out}/{n_trained} trained folds")
print(f"Significantly underperforms EQW : {n_under}/{n_trained} trained folds")

agg_summary = {
    # fold tracking
    "n_folds_configured": len(WALK_FORWARD_FOLDS),
    "n_folds_trained":    n_trained,
    "folds_trained":      list(all_fold_results.keys()),
    "folds_configured":   [f["name"] for f in WALK_FORWARD_FOLDS],
    # version
    "version":            VERSION,
    "risk_profile":       RISK_PROFILE,
    "use_soft_mask":      USE_SOFT_MASK,
    "use_delta_action":   USE_DELTA_ACTION,
    # risk-free rate
    "annual_rfr":         ANNUAL_RISK_FREE_RATE,
    "ann_log_rfr":        ANN_LOG_RFR,
    "rrfr_source":        _rrfr_source,
    # conventions
    "return_convention":   "CAGR = exp(mean_log_return * periods_per_year) - 1",
    "turnover_convention": "full round-trip sum(abs(delta)); divide by 2 for academic one-way",
    "significance_test":   f"HAC Newey-West t-test (lags={HAC_LAGS}) + Wilcoxon",
    "bootstrap_method":    "moving-block bootstrap (non-circular, non-wrapping)",
    # architecture
    "architecture": {
        "encoder_input_dim":  ENCODER_INPUT_DIM,
        "encoder_hidden_dim": ENCODER_HIDDEN_DIM,
        "encoder_output_dim": ENCODER_OUTPUT_DIM,
        "attention_heads":    ATTENTION_HEADS,
        "policy_hidden_dims": POLICY_HIDDEN_DIMS,
        "obs_dim":            OBS_DIM,
        "features_dim":       FEATURES_DIM,
        "n_features":         len(FEATURE_NAMES),
        "feature_names":      FEATURE_NAMES,
    },
    # model assumptions — updated to reflect fixes applied
    "model_assumptions": {
        "1": (
            "NOT FIXABLE without redesign. "
            "portfolio_return = dot(weights_t, sum(daily_log_returns, t→t+21)). "
            "Summing daily log returns is correct for a fixed-weight portfolio "
            "(log-additivity holds). Residual mismatch: weights drift intra-window "
            "as prices move. Fixing this requires a daily-step environment. "
            "Acknowledged as a model limitation."
        ),
        "2": (
            "ADDRESSED. "
            "RL agent can hold up to MAX_CASH_WEIGHT cash; EQW is always fully invested. "
            "Addressed by multi_episode_evaluate_no_cash() which forces cash=0 post-hoc, "
            "and timing_option_sharpe in results which decomposes total Sharpe margin "
            "into selection alpha + timing option value."
        ),
        "3": (
            "PARTIALLY FIXED. "
            "Previous reward: local penalized per-step log return only. "
            "Updated reward: weighted combination of per-step return and a rolling "
            f"Sharpe estimate over the last {SHARPE_REWARD_WINDOW} steps (sharpe_weight="
            f"{REWARD_CONFIGS['sharpe_weight']}). The rolling Sharpe term trains the "
            "policy to be aware of its own return volatility, narrowing the gap with "
            "the Sharpe-based evaluation metric. "
            "Residual gap: rolling Sharpe over a short window != global CAGR/Sharpe "
            "over the full episode. Full elimination requires episodic Sharpe reward "
            "which is incompatible with per-step PPO training."
        ),
        "4": (
            "PARTIALLY FIXED. "
            "Previous: 5 sequential nonlinear projections (delta → mask → cash cap "
            "→ position cap → renormalize → final cap → min-position → turnover cap). "
            "Updated: consolidated to 3 stages via _project_simplex: "
            "(1) action → unconstrained proposed weights, "
            "(2) single combined simplex projection enforcing all weight constraints "
            "simultaneously (mask, cash cap, stock cap, min-position, renormalize, "
            "one correction pass), "
            "(3) turnover cap. "
            "Reduces many-to-one mapping and discontinuities from sequential redistribution. "
            "Residual: turnover cap at stage 3 still creates a discontinuous projection "
            "when binding. Not eliminable without removing the cap entirely."
        ),
    },
    # fixes applied — complete cumulative list
    "fixes_applied": [
        "liquidity_multiplier: 1.3 - 0.3*rel_vol (was always >= 1.0)",
        "position_cap: hard clip after redistribution loop",
        "benchmark_cost: same liquidity-adjusted function as RL",
        "benchmark_init: uniform weights matching RL reset",
        "metrics: CAGR and log-domain RFR throughout",
        "significance: HAC Newey-West replaces naive t-test",
        "bootstrap: moving-block replaces circular np.roll",
        "ir: single-pass non-overlapping computation added",
        "aggregate: reports trained folds only; warns when < 4",
        "delta_action: position-relative deltas replace softmax",
        "min_position: dust positions zeroed and redistributed",
        "final_cap_enforcement: unconditional hard clip fixes AssertionError max=0.200475",
        "cash_ablation: timing option decomposition added to evaluation",
        "ent_coef: raised 0.01 → 0.03 to discourage logit explosion",
        "lgbm_pred_valid: missingness indicator added as feature",
        "lgbm_ablation: zeros both lgbm_pred and lgbm_pred_valid",
        "attention_masking: post-residual mask multiplication prevents leakage",
        "rolling_sharpe_reward: sharpe_weight*rolling_sharpe added to reward (assumption [3])",
        "projection_cascade: consolidated 5-stage pipeline to 3-stage _project_simplex (assumption [4])",
        "model_assumptions: all four assumptions documented with fix status in aggregate JSON",
        "cash_penalty: binary bull/bear gate replacing continuous regime_val multiplier; regime_median computed per-fold on train data only (no look-ahead)",
        "downside_penalty: dropped from 0.8 to 0.0 — redundant with rolling Sharpe reward and primary cause of excess cash hoarding",
        "regime_median: per-fold train-only median passed to all PortfolioEnv instances including signal diagnostic and LGBM ablation envs",
    ],
    # performance
    "rl_mean_sharpe":         float(np.mean(rl_sharpes)),
    "rl_std_sharpe":          float(np.std(rl_sharpes)),
    "eq_mean_sharpe":         float(np.mean(eq_sharpes)),
    "eq_std_sharpe":          float(np.std(eq_sharpes)),
    "rl_no_cash_mean_sharpe": float(np.mean(nc_sharpes)),
    "mean_timing_option":     float(np.mean(timing_vals)),
    "sharpe_margin":          float(np.mean(rl_sharpes) - np.mean(eq_sharpes)),
    "selection_alpha_margin": float(np.mean(nc_sharpes) - np.mean(eq_sharpes)),
    "rl_mean_cagr":           float(np.mean(rl_returns)),
    "eq_mean_cagr":           float(np.mean(eq_returns)),
    "n_outperforms":          int(n_out),
    "n_underperforms":        int(n_under),
}

with open(VERSION_DIR / "aggregate_results.json", "w") as f:
    json.dump(agg_summary, f, indent=2)

print(f"\n-- Final Summary --")
print(f"  VERSION              : {VERSION}")
print(f"  Folds trained        : {list(all_fold_results.keys())} ({n_trained}/{len(WALK_FORWARD_FOLDS)})")
print(f"  RFR                  : {ANNUAL_RISK_FREE_RATE:.4%}  [{_rrfr_source}]")
print(f"  RL mean Sharpe       : {agg_summary['rl_mean_sharpe']:+.3f} ± {agg_summary['rl_std_sharpe']:.3f}")
print(f"  RL no-cash Sharpe    : {agg_summary['rl_no_cash_mean_sharpe']:+.3f}  "
      f"(selection alpha only)")
print(f"  EQW mean Sharpe      : {agg_summary['eq_mean_sharpe']:+.3f} ± {agg_summary['eq_std_sharpe']:.3f}")
print(f"  Sharpe margin (total): {agg_summary['sharpe_margin']:+.3f}")
print(f"  Sharpe margin (stock): {agg_summary['selection_alpha_margin']:+.3f}  "
      f"(cash timing removed)")
print(f"  RL mean CAGR         : {agg_summary['rl_mean_cagr']:+.1f}%")
print(f"  EQW mean CAGR        : {agg_summary['eq_mean_cagr']:+.1f}%")
print(f"  Outperforms          : {n_out}/{n_trained} trained folds")
print(f"\n  Aggregate saved : {VERSION_DIR / 'aggregate_results.json'}")
print(f"\n✅ {VERSION} complete.")


  CROSS-FOLD AGGREGATE — v6_higher_downside_and_hhi_penalties
  1/4 folds trained

  ⚠ WARNING: Only 1 fold(s) trained.
    Std = 0.000 across all metrics — not a meaningful cross-fold result.
    Run all 4 folds before reporting aggregate statistics.

Metric                    RL Mean   RL Std |    EQ Mean   EQ Std
-------------------------------------------------------------------
CAGR                       17.17%    0.00% |     26.55%    0.00%
ann_vol                    14.56%    0.00% |     21.00%    0.00%
sharpe                      0.55    0.00 |      0.75    0.00
sortino                     0.79    0.00 |      0.80    0.00
max_dd                     27.89%    0.00% |     29.75%    0.00%
calmar                      0.62    0.00 |      0.89    0.00
cvar_95                    -8.74%    0.00% |    -15.80%    0.00%

-- Cash Timing Option Summary (model assumption [2]) --
  RL mean Sharpe (with cash)    : +0.547
  RL mean Sharpe (no cash)      : +0.979
  Mean timing option value     

In [ ]:
print("\n-- Signal/Cash Correlation Diagnostic --")
print("(Negative ρ = agent correctly holds more cash in bearish regime ✓)")
for fold_name, model in fold_models.items():
    split    = fold_split_indices[fold_name]
    eval_env = PortfolioEnv(
        feature_tensor      = feature_tensor,
        returns_tensor      = returns_tensor,
        mask_tensor         = mask_tensor,
        market_signal       = market_signal,
        profile_mask_tensor = profile_mask_tensor,
        universe_indices    = universe_indices,
        split_indices       = split,
        risk_profile        = RISK_PROFILE,
        feature_names       = FEATURE_NAMES,
        mode                = "test",
        episode_length      = EPISODE_LENGTH,
        rebalance_days      = REBALANCE_DAYS,
        silent              = True,
        regime_median       = fold_regime_medians[fold_name],  # FIXED
    )
    signals, cash_weights = [], []
    obs, _ = eval_env.reset()
    for _ in range(EPISODE_LENGTH):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, _, truncated, info = eval_env.step(action)
        day_idx = eval_env.start_idx + (eval_env.current_step - 1) * REBALANCE_DAYS
        signals.append(float(market_signal[day_idx, 0]))
        cash_weights.append(info["current_weights"][-1])
        if truncated:
            break
    corr   = np.corrcoef(signals, cash_weights)[0, 1]
    status = ("✓ using signal" if corr < -0.1
              else "✗ not using signal" if corr > 0.1
              else "~ weak")
    print(f"  {fold_name}: signal↔cash ρ = {corr:+.3f}  {status}")


-- Signal/Cash Correlation Diagnostic --
(Negative ρ = agent correctly holds more cash in bearish regime ✓)
  fold_4: signal↔cash ρ = -0.302  ✓ using signal


In [ ]:
from scipy.stats import spearmanr


def collect_lgbm_diagnostic_data(model, env_template, split_indices,
                                  mode="test", episode_length=EPISODE_LENGTH,
                                  max_episodes=MAX_EVAL_EPISODES):
    """Collect PPO weights, LGBM predictions, dates, tickers for diagnostic tests."""
    eval_env     = _make_eval_env(env_template, split_indices, mode, episode_length)
    valid_starts = _sample_valid_starts(eval_env, max_episodes)
    lgbm_idx     = FEATURE_NAMES.index("lgbm_pred")
    rows         = []

    for start_idx in valid_starts:
        obs, _ = eval_env.reset(options={"start_idx": start_idx})
        for _ in range(episode_length):
            day_idx   = eval_env.start_idx + eval_env.current_step * REBALANCE_DAYS
            action, _ = model.predict(obs, deterministic=True)
            obs, _, _, truncated, info = eval_env.step(action)
            weights   = np.array(info["current_weights"][:-1])
            preds     = eval_env.feature_tensor[
                day_idx, eval_env.universe_indices, lgbm_idx
            ]
            mask = eval_env._get_combined_mask(day_idx) > 0
            for i, ticker in enumerate(tickers):
                if not mask[i]:
                    continue
                rows.append({
                    "date":      date_index[day_idx],
                    "ticker":    ticker,
                    "weight":    float(weights[i]),
                    "lgbm_pred": float(preds[i]),
                })
            if truncated:
                break

    return pd.DataFrame(rows)


def lgbm_weight_spearman(df):
    """Test 1: Spearman rank correlation between LGBM prediction and portfolio weight."""
    daily_rhos, daily_pvals = [], []
    for _, grp in df.groupby("date"):
        if len(grp) > 5:
            rho, pval = spearmanr(grp["lgbm_pred"], grp["weight"])
            daily_rhos.append(rho)
            daily_pvals.append(pval)

    rho  = float(np.nanmean(daily_rhos))
    pval = float(np.nanmean(daily_pvals))

    if   rho > 0.50: verdict = "Strong use of LGBM signal"
    elif rho > 0.30: verdict = "Moderate use of LGBM signal"
    elif rho > 0.10: verdict = "Weak use of LGBM signal"
    elif rho > -0.10: verdict = "Mostly ignoring LGBM signal"
    else:             verdict = "Negative relationship"

    print(f"\n{'='*60}")
    print(f"LGBM SIGNAL TEST #1 — Spearman ρ (weight vs prediction)")
    print(f"{'='*60}")
    print(f"  Spearman rho : {rho:.4f}")
    print(f"  p-value      : {pval:.8f}")
    print(f"  Verdict      : {verdict}")
    return {"rho": rho, "pval": pval, "verdict": verdict}


def lgbm_bucket_test(df):
    """Test 2: Compare average weight allocated to top vs bottom LGBM quintile."""
    results = []
    for _, grp in df.groupby("date"):
        if len(grp) < 10:
            continue
        grp    = grp.sort_values("lgbm_pred")
        n      = max(1, len(grp) // 5)
        results.append({
            "top_weight":    grp.iloc[-n:]["weight"].mean(),
            "bottom_weight": grp.iloc[:n]["weight"].mean(),
        })

    results    = pd.DataFrame(results)
    top_avg    = float(results["top_weight"].mean())
    bottom_avg = float(results["bottom_weight"].mean())
    ratio      = top_avg / bottom_avg if bottom_avg > 1e-8 else np.nan

    print(f"\n{'='*60}")
    print(f"LGBM SIGNAL TEST #2 — Top vs Bottom Quintile Allocation")
    print(f"{'='*60}")
    print(f"  Top quintile avg weight    : {top_avg:.4%}")
    print(f"  Bottom quintile avg weight : {bottom_avg:.4%}")
    print(f"  Allocation ratio           : {ratio:.2f}x")
    return {"top_weight": top_avg, "bottom_weight": bottom_avg, "ratio": ratio}


def run_lgbm_ablation_test(model, env_template, split_indices):
    """Test 3: Feature ablation — evaluate with LGBM prediction zeroed out."""
    lgbm_idx          = FEATURE_NAMES.index("lgbm_pred")
    modified_features = feature_tensor.copy()
    modified_features[:, :, lgbm_idx] = 0.0

    ablation_env = PortfolioEnv(
        feature_tensor      = modified_features,
        returns_tensor      = returns_tensor,
        mask_tensor         = mask_tensor,
        market_signal       = market_signal,
        profile_mask_tensor = profile_mask_tensor,
        universe_indices    = universe_indices,
        split_indices       = split_indices,
        risk_profile        = RISK_PROFILE,
        feature_names       = FEATURE_NAMES,
        mode                = "test",
        episode_length      = EPISODE_LENGTH,
        rebalance_days      = REBALANCE_DAYS,
        silent              = True,
        regime_median       = env_template.regime_median,  # FIXED — inherited from template
    )
    valid_starts = _sample_valid_starts(ablation_env, MAX_EVAL_EPISODES)
    ablated_returns = []
    for start_idx in valid_starts:
        obs, _ = ablation_env.reset(options={"start_idx": start_idx})
        for _ in range(EPISODE_LENGTH):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, _, truncated, info = ablation_env.step(action)
            ablated_returns.append(info["net_return"])
            if truncated:
                break

    ablated_arr      = np.array(ablated_returns)
    baseline_returns, _ = multi_episode_evaluate(
        model, env_template, split_indices, "test", EPISODE_LENGTH
    )
    baseline = compute_metrics(baseline_returns, "Original",  print_results=False)
    ablated  = compute_metrics(ablated_arr,       "No LGBM",  print_results=False)

    print(f"\n{'='*60}")
    print(f"LGBM SIGNAL TEST #3 — Feature Ablation")
    print(f"{'='*60}")
    print(f"  Sharpe  Original : {baseline['sharpe']:.3f}")
    print(f"  Sharpe  No LGBM  : {ablated['sharpe']:.3f}  "
          f"(Δ {baseline['sharpe'] - ablated['sharpe']:+.3f})")
    print(f"  CAGR    Original : {baseline['ann_return']*100:.2f}%")
    print(f"  CAGR    No LGBM  : {ablated['ann_return']*100:.2f}%  "
          f"(Δ {(baseline['ann_return'] - ablated['ann_return'])*100:+.2f}%)")
    return {"baseline": baseline, "ablated": ablated}


# Run diagnostics for all trained folds
for fold_name, model in fold_models.items():
    print(f"\n{'#'*80}")
    print(f"  LGBM DIAGNOSTICS — {fold_name}")
    print(f"{'#'*80}")
    split          = fold_split_indices[fold_name]
    diagnostic_df  = collect_lgbm_diagnostic_data(model, _template_env, split, mode="test")
    lgbm_weight_spearman(diagnostic_df)
    lgbm_bucket_test(diagnostic_df)
    run_lgbm_ablation_test(model, _template_env, split)

# %%


################################################################################
  LGBM DIAGNOSTICS — fold_4
################################################################################



LGBM SIGNAL TEST #1 — Spearman ρ (weight vs prediction)
  Spearman rho : 0.0292
  p-value      : 0.42413525
  Verdict      : Mostly ignoring LGBM signal

LGBM SIGNAL TEST #2 — Top vs Bottom Quintile Allocation
  Top quintile avg weight    : 5.9000%
  Bottom quintile avg weight : 5.7148%
  Allocation ratio           : 1.03x

LGBM SIGNAL TEST #3 — Feature Ablation
  Sharpe  Original : 0.547
  Sharpe  No LGBM  : 0.563  (Δ -0.016)
  CAGR    Original : 17.17%
  CAGR    No LGBM  : 17.35%  (Δ -0.18%)


In [ ]:
for fold_name in fold_models:
    alloc_path = VERSION_DIR / fold_name / "portfolio_allocations.csv"
    if not alloc_path.exists():
        continue
    allocations_df = pd.read_csv(alloc_path)

    avg_alloc = (
        allocations_df
        .drop(columns=["episode", "rebalance", "date"])
        .mean()
        .sort_values(ascending=False)
    )
    print(f"\n-- {fold_name} — Top 15 Average Holdings --")
    print(avg_alloc.head(15).to_string())

    latest_alloc = (
        allocations_df.iloc[-1]
        .drop(["episode", "rebalance", "date"])
    )
    latest_alloc = latest_alloc[latest_alloc > 0.001].sort_values(ascending=False)
    print(f"\n-- {fold_name} — Final Portfolio --")
    print(latest_alloc.to_string())


-- fold_4 — Top 15 Average Holdings --
cash    0.244298
EGAL    0.060690
JUFO    0.057549
EGCH    0.057520
EAST    0.049354
ORWE    0.047793
FWRY    0.038154
EFID    0.037123
ARCC    0.034292
GBCO    0.034252
EFIH    0.034198
AMOC    0.028801
EMFD    0.027252
RAYA    0.025256
ADIB    0.021441

-- fold_4 — Final Portfolio --
ARCC     0.168362
EFID     0.168188
ABUK     0.160289
BTFH     0.126943
VLMRA    0.126895
cash     0.117112
FWRY     0.032207
EGAL     0.026655
OIH      0.019555
JUFO     0.014639
MCQE     0.014639
GBCO     0.009518
EMFD     0.006134
PHDC     0.005777
